# RouteRanker – Module 2: Feature Engineering

## 1. Introduction

RouteRanker is an AI-powered public transit route optimization project that uses General Transit Feed Specification (GTFS) data to understand scheduled transportation services.

Module 2 focuses on transforming raw GTFS data into meaningful features for route analysis, comparison, and future machine learning tasks.

## 2. Objectives

- Extract time-based features from scheduled trip data.
- Calculate trip counts and route-level summaries.
- Generate route profiles and hourly departure summaries.
- Calculate scheduled headway between consecutive departures.
- Perform data quality checks and document assumptions and limitations.

## 3. Dataset

This module uses GTFS data collected during Module 1 for five cities: Ahmedabad, Chennai, Hyderabad, Mumbai, and Pune.

The data includes route, trip, stop, and scheduled stop-time information where available.

**Important limitation:** GTFS schedule data describes planned services. It does not provide actual passenger counts or prove that a route is overcrowded.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

cities = ["ahmedabad", "chennai", "hyderabad", "mumbai", "pune"]

In [2]:
cities = ["ahmedabad", "chennai", "hyderabad", "mumbai", "pune"]

for city in cities:
    file_path = PROCESSED_DIR / f"{city}_trip_time_features.csv"

    if file_path.exists():
        df = pd.read_csv(file_path)
        print(f"{city.title()}: {df.shape[0]} rows, {df.shape[1]} columns")
        print("Columns:", list(df.columns))
        print("-" * 60)
    else:
        print(f"File not found: {file_path}")

Ahmedabad: 14323 rows, 10 columns
Columns: ['city', 'trip_id', 'route_id', 'service_id', 'departure_time', 'departure_time_seconds', 'service_hour', 'clock_hour', 'minute', 'time_period']
------------------------------------------------------------
Chennai: 48435 rows, 10 columns
Columns: ['city', 'trip_id', 'route_id', 'service_id', 'departure_time', 'departure_time_seconds', 'service_hour', 'clock_hour', 'minute', 'time_period']
------------------------------------------------------------
Hyderabad: 44892 rows, 10 columns
Columns: ['city', 'trip_id', 'route_id', 'service_id', 'departure_time', 'departure_time_seconds', 'service_hour', 'clock_hour', 'minute', 'time_period']
------------------------------------------------------------
Mumbai: 99080 rows, 10 columns
Columns: ['city', 'trip_id', 'route_id', 'service_id', 'departure_time', 'departure_time_seconds', 'service_hour', 'clock_hour', 'minute', 'time_period']
------------------------------------------------------------
Pune: 101

In [3]:
for city in cities:
    file_path = PROCESSED_DIR / f"{city}_trip_headway_features.csv"

    if file_path.exists():
        df = pd.read_csv(file_path)
        print(f"{city.title()}: {df.shape[0]} rows, {df.shape[1]} columns")
        print("Headway column available:", "headway_minutes" in df.columns)
        print("Missing headways:", df["headway_minutes"].isna().sum()
              if "headway_minutes" in df.columns else "Column missing")
        print("-" * 60)
    else:
        print(f"File not found: {file_path}")

Ahmedabad: 14323 rows, 11 columns
Headway column available: True
Missing headways: 633
------------------------------------------------------------
Chennai: 48435 rows, 11 columns
Headway column available: True
Missing headways: 4732
------------------------------------------------------------
Hyderabad: 44892 rows, 11 columns
Headway column available: True
Missing headways: 1031
------------------------------------------------------------
Mumbai: 99080 rows, 11 columns
Headway column available: True
Missing headways: 612
------------------------------------------------------------
Pune: 10128 rows, 11 columns
Headway column available: True
Missing headways: 315
------------------------------------------------------------


In [4]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORTS_DIR = PROJECT_DIR / "reports"

cities = ["ahmedabad", "chennai", "hyderabad", "mumbai", "pune"]

In [5]:
for city in cities:
    file_path = REPORTS_DIR / f"{city}_route_profile.csv"

    if file_path.exists():
        df = pd.read_csv(file_path)
        print(f"{city.title()}: {df.shape[0]} routes, {df.shape[1]} columns")
        print("Columns:", list(df.columns))
        print("-" * 60)
    else:
        print(f"File not found: {file_path}")

Ahmedabad: 639 routes, 20 columns
Columns: ['route_id', 'route_short_name', 'route_long_name', 'route_type', 'scheduled_trip_count', 'stop_count', 'estimated_straight_line_route_km', 'trips_with_distance', 'operates_monday', 'operates_tuesday', 'operates_wednesday', 'operates_thursday', 'operates_friday', 'operates_saturday', 'operates_sunday', 'operating_day_count', 'city', 'start_service_hour', 'end_service_hour', 'trips_with_valid_departure']
------------------------------------------------------------
Chennai: 5402 routes, 20 columns
Columns: ['route_id', 'route_short_name', 'route_long_name', 'route_type', 'scheduled_trip_count', 'stop_count', 'estimated_straight_line_route_km', 'trips_with_distance', 'operates_monday', 'operates_tuesday', 'operates_wednesday', 'operates_thursday', 'operates_friday', 'operates_saturday', 'operates_sunday', 'operating_day_count', 'city', 'start_service_hour', 'end_service_hour', 'trips_with_valid_departure']
----------------------------------------

In [6]:
feature_dict_path = REPORTS_DIR / "module2_feature_dictionary.csv"

if feature_dict_path.exists():
    feature_dict = pd.read_csv(feature_dict_path)
    print("Feature dictionary shape:", feature_dict.shape)
    display(feature_dict.head(10))
    print("\nColumn names:", list(feature_dict.columns))
else:
    print("Feature dictionary file not found:", feature_dict_path)

Feature dictionary shape: (19, 4)


,feature,description,source_or_method,data_type
0,city,City containing the route,Folder name,Categorical
1,route_id,GTFS route identifier,routes.txt,Identifier
2,route_short_name,Short public route name when provided,routes.txt,Categorical
3,route_long_name,Long route name when provided,routes.txt,Categorical
4,route_type,GTFS transport mode code,routes.txt,Categorical/code
5,scheduled_trip_count,Number of distinct scheduled trips,trips.txt,Integer
6,stop_count,Number of distinct stops referenced by route t...,stop_times.txt + trips.txt,Integer
7,scheduled_departures,Distinct scheduled trip departures in a servic...,First stop per trip,Integer
8,headway_minutes,Minutes between consecutive scheduled first-st...,Difference between consecutive departure_time_...,Numeric minutes
9,service_hour,GTFS service-time hour; may exceed 23,departure_time,Integer



Column names: ['feature', 'description', 'source_or_method', 'data_type']


In [7]:
quality_path = REPORTS_DIR / "module2_data_quality_report.csv"

if quality_path.exists():
    quality_report = pd.read_csv(quality_path)
    print("Data quality report shape:", quality_report.shape)
    display(quality_report.head(15))
    print("\nColumn names:", list(quality_report.columns))
else:
    print("Data quality report not found:", quality_path)

Data quality report shape: (152, 7)


,city,file,check,column,count,total_rows,percent
0,ahmedabad,routes.txt,missing_values,route_id,0,639,0.0
1,ahmedabad,routes.txt,missing_values,agency_id,0,639,0.0
2,ahmedabad,routes.txt,missing_values,route_short_name,0,639,0.0
3,ahmedabad,routes.txt,missing_values,route_long_name,0,639,0.0
4,ahmedabad,routes.txt,missing_values,route_type,0,639,0.0
5,ahmedabad,routes.txt,duplicate_key_rows,route_id,0,639,0.0
6,ahmedabad,trips.txt,missing_values,route_id,0,14323,0.0
7,ahmedabad,trips.txt,missing_values,service_id,0,14323,0.0
8,ahmedabad,trips.txt,missing_values,trip_id,0,14323,0.0
9,ahmedabad,trips.txt,missing_values,trip_headsign,0,14323,0.0



Column names: ['city', 'file', 'check', 'column', 'count', 'total_rows', 'percent']


In [8]:
assumptions_path = REPORTS_DIR / "module2_assumptions_and_limitations.md"

if assumptions_path.exists():
    print(assumptions_path.read_text(encoding="utf-8"))
else:
    print("Assumptions file not found:", assumptions_path)

# RouteRanker Module 2: Assumptions and Limitations

1. GTFS files describe scheduled transit service, not actual passenger counts.
2. Scheduled trips and departures must not be interpreted as passenger occupancy or crowding.
3. A trip's departure features use the first stop in stop_sequence.
4. GTFS times above 24:00 are retained as extended service hours.
5. Clock hour is service hour modulo 24; time-period labels use the clock hour.
6. Operating days are derived from matching trips.service_id to calendar.service_id.
7. Calendar exceptions are not applied unless calendar_dates.txt is available and implemented.
8. Missing or invalid data are reported; they are not automatically replaced with zero.
9. Route IDs and trip IDs are identifiers, not numeric measurements.
10. No shapes.txt file was found in the data directory. Route distances, where available,
    are approximate straight-line distances along consecutive stop coordinates. They are
    not road distances and can be inaccurate

In [9]:
print("RouteRanker — Module 2: Feature Engineering")
print("=" * 50)

print("\nCities processed:", len(cities))
print("Cities:", ", ".join(city.title() for city in cities))

print("\nDeliverables:")
deliverables = [
    REPORTS_DIR / "module2_feature_dictionary.csv",
    REPORTS_DIR / "module2_data_quality_report.csv",
    REPORTS_DIR / "module2_assumptions_and_limitations.md",
]

for path in deliverables:
    print(f"{'Available' if path.exists() else 'Missing'}: {path.name}")

print("\nFeature datasets:")
for city in cities:
    files = [
        PROCESSED_DIR / f"{city}_trip_time_features.csv",
        PROCESSED_DIR / f"{city}_trip_headway_features.csv",
        REPORTS_DIR / f"{city}_route_profile.csv",
    ]
    status = all(path.exists() for path in files)
    print(f"{city.title()}: {'Complete' if status else 'Check missing files'}")

print("\nLimitation: Features describe scheduled service, not actual passenger crowding.")

RouteRanker — Module 2: Feature Engineering

Cities processed: 5
Cities: Ahmedabad, Chennai, Hyderabad, Mumbai, Pune

Deliverables:
Available: module2_feature_dictionary.csv
Available: module2_data_quality_report.csv
Available: module2_assumptions_and_limitations.md

Feature datasets:
Ahmedabad: Complete
Chennai: Complete
Hyderabad: Complete
Mumbai: Complete
Pune: Complete

Limitation: Features describe scheduled service, not actual passenger crowding.


## 2. Feature Engineering Implementation

The feature-engineering process is implemented in `notebook/module2_feature_engineering.py`.

The script processes GTFS data for Ahmedabad, Chennai, Hyderabad, Mumbai, and Pune. It extracts scheduled trip departure information, derives time-based features, calculates scheduled headways, creates route-level profiles, and generates data-quality reports.

The implementation also creates a feature dictionary and documents assumptions and limitations. Headways represent differences between scheduled departures, not observed bus intervals or passenger crowding. Route distances derived from stop coordinates are approximate straight-line estimates, not road distances.

In [10]:
from pathlib import Path

SCRIPT_PATH = PROJECT_DIR / "notebook" / "module2_feature_engineering.py"

if SCRIPT_PATH.exists():
    source_code = SCRIPT_PATH.read_text(encoding="utf-8")
    print("Implementation script:", SCRIPT_PATH)
    print("Script size:", len(source_code), "characters")
    print("\nMain functions found:")
    for function_name in [
        "read_gtfs",
        "parse_gtfs_time",
        "time_period",
        "haversine_km",
        "add_quality",
        "process_city",
        "main",
    ]:
        print(f"- {function_name}:",
              f"{'Available' if f'def {function_name}(' in source_code else 'Not found'}")
else:
    print("Implementation script not found:", SCRIPT_PATH)

Implementation script: C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\notebook\module2_feature_engineering.py
Script size: 22146 characters

Main functions found:
- read_gtfs: Available
- parse_gtfs_time: Available
- time_period: Available
- haversine_km: Available
- add_quality: Available
- process_city: Available
- main: Available


In [11]:
from IPython.display import Code, display

display(Code(source_code, language="python"))

from pathlib import Path
import math
import pandas as pd

# --------------------------------------------------
# ROUTERANKER - MODULE 2: FEATURE ENGINEERING
# --------------------------------------------------

BASE_DIR = Path(__file__).resolve().parent.parent
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
REPORTS_DIR = BASE_DIR / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

CITIES = ["ahmedabad", "chennai", "hyderabad", "mumbai", "pune"]

DAY_COLUMNS = [
    "monday", "tuesday", "wednesday", "thursday",
    "friday", "saturday", "sunday"
]


def read_gtfs(city, filename, required=True):
    """Read a GTFS text file while preserving IDs as strings."""
    path = RAW_DIR / city / filename

    if not path.exists():
        if required:
            raise FileNotFoundError(f"Missing file: {path}")
        return None

    return pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False,
        low_memory=False
    )


def parse_gtfs_time(value):
    """
    Convert GTFS HH:MM:SS into seconds.
    Supports extended hours such as 25:10:00.
    Returns None for invalid or missing values.
    """
    try:
        parts = str(value).strip().split(":")
        if len(parts) != 3:
            return None

        hours, minutes, seconds = map(int, parts)

        if hours < 0 or not (0 <= minutes < 60):
            return None
        if not (0 <= seconds < 60):
            return None

        return hours * 3600 + minutes * 60 + seconds
    except (ValueError, TypeError):
        return None


def time_period(hour):
    """Classify the clock hour into a broad time period."""
    if 5 <= hour <= 11:
        return "Morning"
    elif 12 <= hour <= 16:
        return "Afternoon"
    elif 17 <= hour <= 20:
        return "Evening"
    return "Night"


def haversine_km(lat1, lon1, lat2, lon2):
    """Calculate approximate straight-line distance between coordinates."""
    values = [lat1, lon1, lat2, lon2]

    if any(pd.isna(v) for v in values):
        return None

    if not (-90 <= lat1 <= 90 and -90 <= lat2 <= 90):
        return None
    if not (-180 <= lon1 <= 180 and -180 <= lon2 <= 180):
        return None

    radius = 6371.0088

    lat1, lon1, lat2, lon2 = map(
        math.radians, values
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    a = min(1.0, max(0.0, a))

    return 2 * radius * math.asin(math.sqrt(a))


def add_quality(rows, city, filename, df, key_columns=None):
    """Record missing values, duplicate keys and row counts."""
    for column in df.columns:
        missing = df[column].isna() | df[column].astype(str).str.strip().eq("")

        rows.append({
            "city": city,
            "file": filename,
            "check": "missing_values",
            "column": column,
            "count": int(missing.sum()),
            "total_rows": len(df),
            "percent": round(missing.mean() * 100, 2) if len(df) else 0
        })

    if key_columns:
        available = [c for c in key_columns if c in df.columns]

        if available:
            duplicate_mask = df.duplicated(
                subset=available, keep=False
            )

            rows.append({
                "city": city,
                "file": filename,
                "check": "duplicate_key_rows",
                "column": ",".join(available),
                "count": int(duplicate_mask.sum()),
                "total_rows": len(df),
                "percent": round(duplicate_mask.mean() * 100, 2)
                if len(df) else 0
            })


def process_city(city, quality_rows):
    print(f"\nProcessing {city.title()}...")

    routes = read_gtfs(city, "routes.txt")
    trips = read_gtfs(city, "trips.txt")
    stop_times = read_gtfs(city, "stop_times.txt")
    stops = read_gtfs(city, "

## 3. Module 2 – Results and Observations

### 3.1 Feature Engineering Results

Feature engineering was performed on GTFS public transit data from five Indian cities: Ahmedabad, Chennai, Hyderabad, Mumbai, and Pune. The process generated trip-time features, scheduled headway features, hourly departure summaries, and route-level profiles.

### 3.2 Features Generated

- **Trip-time features:** Departure time, service hour, clock hour, and time period.
- **Headway features:** Time difference between consecutive scheduled departures for the same route and service.
- **Hourly departure features:** Scheduled departures grouped by route and hour.
- **Route-level features:** Scheduled trip count, stop count, approximate straight-line route distance, operating-day indicators, and service-hour information.
- **Data-quality checks:** Missing values and other data-quality issues were documented in reports.

### 3.3 Key Observations

The generated features provide a foundation for comparing scheduled service patterns across routes and cities. Headway and hourly departure features can help identify routes or time periods with relatively frequent or infrequent scheduled service.

The distance values are approximate straight-line estimates based on stop coordinates, not actual road distances. Since passenger-count data is unavailable, the current analysis does not directly measure passenger demand or actual crowding.

### 3.4 Conclusion

Module 2 successfully prepared schedule-based features and route profiles for the five selected cities. These outputs can support the next stage of RouteRanker: exploratory analysis, route comparison, and development of schedule-based service improvement recommendations.

**Note:** The results describe scheduled public transit service. They should not be interpreted as verified real-world passenger occupancy or traffic conditions.

In [12]:

import pandas as pd
from pathlib import Path
from IPython.display import display

cities = ["ahmedabad", "chennai", "hyderabad", "mumbai", "pune"]

results = []

for city in cities:
    profile_path = REPORTS_DIR / f"{city}_route_profile.csv"
    trip_path = PROCESSED_DIR / f"{city}_trip_time_features.csv"
    headway_path = PROCESSED_DIR / f"{city}_trip_headway_features.csv"


    if not profile_path.exists():
        print(f"Missing route profile: {profile_path}")
        continue

    profile = pd.read_csv(profile_path)
    trip_features = pd.read_csv(trip_path) if trip_path.exists() else None
    headway_features = pd.read_csv(headway_path) if headway_path.exists() else None

    results.append({
        "City": city.title(),
        "Routes in profile": len(profile),
        "Scheduled trips in profile": int(
            profile["scheduled_trip_count"].fillna(0).sum()
        ),
        "Trip-time feature rows": (
            len(trip_features) if trip_features is not None else None
        ),
        "Headway feature rows": (
            len(headway_features) if headway_features is not None else None
        ),
    })

results_df = pd.DataFrame(results)

print("RouteRanker — Module 2: Actual Output Summary")
display(results_df)


RouteRanker — Module 2: Actual Output Summary


,City,Routes in profile,Scheduled trips in profile,Trip-time feature rows,Headway feature rows
0,Ahmedabad,639,14323,14323,14323
1,Chennai,5402,48435,48435,48435
2,Hyderabad,1031,44892,44892,44892
3,Mumbai,935,99080,99080,99080
4,Pune,315,10128,10128,10128


In [13]:
print("Headway files found:")
for path in sorted(PROCESSED_DIR.glob("*headway*.csv")):
    print(path.name)

Headway files found:
ahmedabad_trip_headway_features.csv
chennai_trip_headway_features.csv
headway_validation_report.csv
hyderabad_trip_headway_features.csv
mumbai_trip_headway_features.csv
pune_trip_headway_features.csv


### Interpretation of Results

RouteRanker processed GTFS schedule data for five Indian cities, generating 216,858 trip-time feature records and 216,858 headway feature records. The route profiles contain 8,322 route entries across the five city datasets.

Mumbai has the highest number of scheduled trips among the selected datasets, with 99,080 trips, while Pune has 10,128. These results describe the supplied GTFS schedules and should not be interpreted as actual passenger counts or verified crowding levels.

The generated features provide a foundation for comparing scheduled service frequency, operating periods, and route characteristics in the next stage of RouteRanker.

In [14]:

import pandas as pd

cities = ["ahmedabad", "chennai", "hyderabad", "mumbai", "pune"]
headway_results = []

for city in cities:
    path = PROCESSED_DIR / f"{city}_trip_headway_features.csv"

    if not path.exists():
        print(f"File not found: {path.name}")
        continue

    df = pd.read_csv(path)

    if "headway_minutes" not in df.columns:
        print(f"Missing headway_minutes column: {path.name}")
        continue

    headway = pd.to_numeric(df["headway_minutes"], errors="coerce")

    headway_results.append({
        "City": city.title(),
        "Total rows": len(df),
        "Valid headways": int(headway.notna().sum()),
        "Missing headways": int(headway.isna().sum()),
        "Zero headways": int(headway.eq(0).sum()),
        "Negative headways": int(headway.lt(0).sum()),
        "Minimum headway (min)": headway.min(),
        "Maximum headway (min)": headway.max()
    })

headway_validation = pd.DataFrame(headway_results)

display(headway_validation.round(2))


,City,Total rows,Valid headways,Missing headways,Zero headways,Negative headways,Minimum headway (min),Maximum headway (min)
0,Ahmedabad,14323,13690,633,933,0,0.0,795.0
1,Chennai,48435,43703,4732,1005,0,0.0,1305.0
2,Hyderabad,44892,43861,1031,13462,0,0.0,1500.0
3,Mumbai,99080,98468,612,9257,0,0.0,1067.0
4,Pune,10128,9813,315,1507,0,0.0,1060.0


In [15]:
import pandas as pd

city = "hyderabad"
path = PROCESSED_DIR / f"{city}_trip_headway_features.csv"

df = pd.read_csv(path)

print("Columns:")
print(df.columns.tolist())

print("\nFirst 10 negative-headway records:")

negative = df[pd.to_numeric(
    df["headway_minutes"], errors="coerce"
) < 0]

display(negative.head(10))

Columns:
['city', 'trip_id', 'route_id', 'service_id', 'departure_time', 'departure_time_seconds', 'service_hour', 'clock_hour', 'minute', 'time_period', 'headway_minutes']

First 10 negative-headway records:


,city,trip_id,route_id,service_id,departure_time,departure_time_seconds,service_hour,clock_hour,minute,time_period,headway_minutes


In [16]:
import pandas as pd

city = "hyderabad"
path = PROCESSED_DIR / f"{city}_trip_headway_features.csv"

df = pd.read_csv(path)
df["headway_minutes"] = pd.to_numeric(
    df["headway_minutes"], errors="coerce"
)

negative = df[df["headway_minutes"] < 0]

print("Total negative records:", len(negative))
print("\nNegative headway range:")
print(negative["headway_minutes"].describe())

print("\nSample negative records:")
print(
    negative[
        [
            "trip_id",
            "route_id",
            "service_id",
            "departure_time",
            "departure_time_seconds",
            "headway_minutes",
        ]
    ].head(10).to_string(index=False)
)

Total negative records: 0

Negative headway range:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: headway_minutes, dtype: float64

Sample negative records:
Empty DataFrame
Columns: [trip_id, route_id, service_id, departure_time, departure_time_seconds, headway_minutes]
Index: []


In [17]:
import pandas as pd

cities = ["ahmedabad", "chennai", "hyderabad", "mumbai", "pune"]
results = []

for city in cities:
    path = PROCESSED_DIR / f"{city}_trip_headway_features.csv"
    df = pd.read_csv(path)

    headway = pd.to_numeric(df["headway_minutes"], errors="coerce")

    results.append({
        "City": city.title(),
        "Total rows": len(df),
        "Valid headways": int(headway.notna().sum()),
        "Missing headways": int(headway.isna().sum()),
        "Zero headways": int(headway.eq(0).sum()),
        "Negative headways": int(headway.lt(0).sum()),
        "Minimum headway": headway.min(),
        "Maximum headway": headway.max()
    })

validation_df = pd.DataFrame(results)
display(validation_df.round(2))


,City,Total rows,Valid headways,Missing headways,Zero headways,Negative headways,Minimum headway,Maximum headway
0,Ahmedabad,14323,13690,633,933,0,0.0,795.0
1,Chennai,48435,43703,4732,1005,0,0.0,1305.0
2,Hyderabad,44892,43861,1031,13462,0,0.0,1500.0
3,Mumbai,99080,98468,612,9257,0,0.0,1067.0
4,Pune,10128,9813,315,1507,0,0.0,1060.0


In [18]:
import pandas as pd

path = PROCESSED_DIR / "hyderabad_trip_headway_features.csv"
df_check = pd.read_csv(path)

h = pd.to_numeric(df_check["headway_minutes"], errors="coerce")

print("File:", path.name)
print("Rows:", len(df_check))
print("Negative values:", int((h < 0).sum()))
print("Zero values:", int((h == 0).sum()))
print("Missing values:", int(h.isna().sum()))
print("Minimum:", h.min())
print("Maximum:", h.max())

File: hyderabad_trip_headway_features.csv
Rows: 44892
Negative values: 0
Zero values: 13462
Missing values: 1031
Minimum: 0.0
Maximum: 1500.0


In [19]:
import pandas as pd

cities = ["ahmedabad", "chennai", "hyderabad", "mumbai", "pune"]
results = []

for city in cities:
    path = PROCESSED_DIR / f"{city}_trip_headway_features.csv"
    df = pd.read_csv(path)
    h = pd.to_numeric(df["headway_minutes"], errors="coerce")

    results.append({
        "City": city.title(),
        "Total rows": len(df),
        "Missing": int(h.isna().sum()),
        "Zero": int(h.eq(0).sum()),
        "Negative": int(h.lt(0).sum()),
        "Minimum": h.min(),
        "Maximum": h.max()
    })

display(pd.DataFrame(results))

,City,Total rows,Missing,Zero,Negative,Minimum,Maximum
0,Ahmedabad,14323,633,933,0,0.0,795.0
1,Chennai,48435,4732,1005,0,0.0,1305.0
2,Hyderabad,44892,1031,13462,0,0.0,1500.0
3,Mumbai,99080,612,9257,0,0.0,1067.0
4,Pune,10128,315,1507,0,0.0,1060.0


In [20]:
for city in ["ahmedabad", "chennai", "hyderabad", "mumbai", "pune"]:
    path = PROCESSED_DIR / f"{city}_trip_headway_features.csv"
    df = pd.read_csv(path)
    h = pd.to_numeric(df["headway_minutes"], errors="coerce")

    print(f"\n{city.title()}")
    print(f"  Total rows:       {len(df)}")
    print(f"  Missing:          {h.isna().sum()}")
    print(f"  Zero:             {h.eq(0).sum()}")
    print(f"  Negative:         {h.lt(0).sum()}")
    print(f"  Minimum:          {h.min()}")
    print(f"  Maximum:          {h.max()}")


Ahmedabad
  Total rows:       14323
  Missing:          633
  Zero:             933
  Negative:         0
  Minimum:          0.0
  Maximum:          795.0

Chennai
  Total rows:       48435
  Missing:          4732
  Zero:             1005
  Negative:         0
  Minimum:          0.0
  Maximum:          1305.0

Hyderabad
  Total rows:       44892
  Missing:          1031
  Zero:             13462
  Negative:         0
  Minimum:          0.0
  Maximum:          1500.0

Mumbai
  Total rows:       99080
  Missing:          612
  Zero:             9257
  Negative:         0
  Minimum:          0.0
  Maximum:          1067.0

Pune
  Total rows:       10128
  Missing:          315
  Zero:             1507
  Negative:         0
  Minimum:          0.0
  Maximum:          1060.0


### Headway Validation – Results and Observations

The headway validation was performed on scheduled trip data from Ahmedabad, Chennai, Hyderabad, Mumbai, and Pune. Across all five cities, 216,858 records were checked, and no negative headway values were found.

Missing headway values were observed for 7,323 records. These are expected for the first scheduled departure within a route and service group, where no previous departure is available for comparison.

Zero headways were observed in 26,164 records. These indicate consecutive records with equal scheduled departure times within the grouping used by the feature-engineering process. These records require further investigation to determine whether they represent valid simultaneous departures or duplicate schedule entries.

The maximum calculated headway varied across the datasets, from 795 minutes in Ahmedabad to 1,500 minutes in Hyderabad. These values describe the supplied schedules and should be interpreted carefully, as long gaps may occur between scheduled trips or service periods.

**Conclusion:** The validation found no negative headway values. Missing and zero headways were retained for further analysis rather than automatically modified. All headway features represent scheduled service patterns, not real-time bus intervals or passenger crowding.

In [21]:
import pandas as pd

path = PROCESSED_DIR / "hyderabad_trip_headway_features.csv"
df = pd.read_csv(path)

zero_headways = df[
    pd.to_numeric(df["headway_minutes"], errors="coerce") == 0
]

print("Total zero-headway records:", len(zero_headways))
print("Unique routes:", zero_headways["route_id"].nunique())
print("Unique trips:", zero_headways["trip_id"].nunique())

print("\nSample records:")
print(
    zero_headways[
        [
            "trip_id",
            "route_id",
            "service_id",
            "departure_time",
            "departure_time_seconds",
            "headway_minutes",
        ]
    ].head(10).to_string(index=False)
)


Total zero-headway records: 13462
Unique routes: 393
Unique trips: 13462

Sample records:
 trip_id route_id service_id departure_time  departure_time_seconds  headway_minutes
41748569        1    MTWTFSS       04:10:00                   15000              0.0
41735584        1    MTWTFSS       05:20:00                   19200              0.0
41748571        1    MTWTFSS       05:30:00                   19800              0.0
41735585        1    MTWTFSS       06:00:00                   21600              0.0
41748294        1    MTWTFSS       06:20:00                   22800              0.0
41748295        1    MTWTFSS       07:00:00                   25200              0.0
41748300        1    MTWTFSS       10:30:00                   37800              0.0
41751526        1    MTWTFSS       11:30:00                   41400              0.0
41756636        1    MTWTFSS       12:00:00                   43200              0.0
41729880        1    MTWTFSS       12:10:00                 

In [22]:
import pandas as pd

path = PROCESSED_DIR / "hyderabad_trip_headway_features.csv"
df = pd.read_csv(path)

zero = df[pd.to_numeric(df["headway_minutes"], errors="coerce") == 0].copy()

same_time_counts = (
    df.groupby(
        ["route_id", "service_id", "departure_time_seconds"]
    )["trip_id"]
    .transform("count")
)

zero_indices = zero.index
zero["Trips at same route/service/time"] = same_time_counts.loc[zero_indices]

print("Zero-headway records:", len(zero))
print(
    "With multiple trips at the same scheduled time:",
    int((zero["Trips at same route/service/time"] > 1).sum())
)
print(
    "With only one trip at that scheduled time:",
    int((zero["Trips at same route/service/time"] == 1).sum())
)

print("\nExamples:")
print(
    zero[
        [
            "trip_id",
            "route_id",
            "service_id",
            "departure_time",
            "Trips at same route/service/time",
        ]
    ].head(10).to_string(index=False)
)

Zero-headway records: 13462
With multiple trips at the same scheduled time: 13462
With only one trip at that scheduled time: 0

Examples:
 trip_id route_id service_id departure_time  Trips at same route/service/time
41748569        1    MTWTFSS       04:10:00                                 2
41735584        1    MTWTFSS       05:20:00                                 2
41748571        1    MTWTFSS       05:30:00                                 2
41735585        1    MTWTFSS       06:00:00                                 2
41748294        1    MTWTFSS       06:20:00                                 2
41748295        1    MTWTFSS       07:00:00                                 2
41748300        1    MTWTFSS       10:30:00                                 2
41751526        1    MTWTFSS       11:30:00                                 2
41756636        1    MTWTFSS       12:00:00                                 2
41729880        1    MTWTFSS       12:10:00                                 2


In [23]:
from pathlib import Path

script_path = Path("..") / "notebook" / "module2_feature_engineering.py"

print("Script path:", script_path.resolve())
print("Script exists:", script_path.exists())

if script_path.exists():
    lines = script_path.read_text(encoding="utf-8").splitlines()

    for i, line in enumerate(lines):
        if "headway_minutes" in line:
            start = max(0, i - 5)
            end = min(len(lines), i + 6)

            print(f"\n--- Lines {start + 1} to {end} ---")
            for j in range(start, end):
                print(f"{j + 1}: {lines[j]}")

Script path: C:\Users\LENOVO\OneDrive\Desktop\notebook\module2_feature_engineering.py
Script exists: False


In [24]:
from pathlib import Path

script_path = Path("notebook") / "module2_feature_engineering.py"

print("Current folder:", Path.cwd())
print("Script path:", script_path.resolve())
print("Script exists:", script_path.exists())


Current folder: C:\Users\LENOVO\OneDrive\Desktop\RouteRanker
Script path: C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\notebook\module2_feature_engineering.py
Script exists: True


In [25]:
lines = script_path.read_text(encoding="utf-8").splitlines()

for i, line in enumerate(lines):
    if "headway_minutes" in line:
        start = max(0, i - 5)
        end = min(len(lines), i + 6)

        print(f"\n--- Lines {start + 1} to {end} ---")
        for j in range(start, end):
            print(f"{j + 1}: {lines[j]}")


--- Lines 330 to 340 ---
330:     # Headways are grouped by route and service_id.
331:     headway_features = departures.sort_values(
332:         ["route_id", "service_id", "departure_time_seconds"]
333:     ).copy()
334: 
335:     headway_features["headway_minutes"] = (
336:         headway_features.groupby(
337:             ["route_id", "service_id"]
338:         )["departure_time_seconds"].diff() / 60
339:     )
340: 

--- Lines 601 to 611 ---
601:         ["route_long_name", "Long route name when provided", "routes.txt", "Categorical"],
602:         ["route_type", "GTFS transport mode code", "routes.txt", "Categorical/code"],
603:         ["scheduled_trip_count", "Number of distinct scheduled trips", "trips.txt", "Integer"],
604:         ["stop_count", "Number of distinct stops referenced by route trips", "stop_times.txt + trips.txt", "Integer"],
605:         ["scheduled_departures", "Distinct scheduled trip departures in a service hour", "First stop per trip", "Integer"],
606: [

In [26]:
import pandas as pd

path = PROCESSED_DIR / "hyderabad_trip_headway_features.csv"
df = pd.read_csv(path)

df["departure_time_seconds"] = pd.to_numeric(
    df["departure_time_seconds"], errors="coerce"
)
df["headway_minutes"] = pd.to_numeric(
    df["headway_minutes"], errors="coerce"
)

df = df.sort_values(
    ["route_id", "service_id", "departure_time_seconds"]
).copy()

df["previous_departure_seconds"] = (
    df.groupby(["route_id", "service_id"])["departure_time_seconds"]
    .shift(1)
)

df["expected_headway_minutes"] = (
    df["departure_time_seconds"] - df["previous_departure_seconds"]
) / 60

zero = df[df["headway_minutes"] == 0]

print("Zero-headway records:", len(zero))
print(
    "Zero headways matching consecutive departures:",
    int((zero["expected_headway_minutes"] == 0).sum())
)
print(
    "Zero headways not matching consecutive departures:",
    int((zero["expected_headway_minutes"] != 0).sum())
)

print("\nSample verification:")
print(
    zero[
        [
            "route_id",
            "service_id",
            "departure_time",
            "previous_departure_seconds",
            "departure_time_seconds",
            "headway_minutes",
            "expected_headway_minutes",
        ]
    ].head(10).to_string(index=False)
)


Zero-headway records: 13462
Zero headways matching consecutive departures: 13462
Zero headways not matching consecutive departures: 0

Sample verification:
route_id service_id departure_time  previous_departure_seconds  departure_time_seconds  headway_minutes  expected_headway_minutes
       1    MTWTFSS       04:10:00                     15000.0                   15000              0.0                       0.0
       1    MTWTFSS       05:20:00                     19200.0                   19200              0.0                       0.0
       1    MTWTFSS       05:30:00                     19800.0                   19800              0.0                       0.0
       1    MTWTFSS       06:00:00                     21600.0                   21600              0.0                       0.0
       1    MTWTFSS       06:20:00                     22800.0                   22800              0.0                       0.0
       1    MTWTFSS       07:00:00                     25200.0  

### Headway Calculation Verification

The headway calculation was verified using the Hyderabad scheduled trip dataset. The saved headway values were compared with independently recalculated differences between consecutive departure times within each route and service group.

All 13,462 zero-headway records matched the independently calculated values, with zero mismatches. This confirms that the zero values were calculated consistently for the checked records. No negative headways were found in the Hyderabad dataset.

Zero headways indicate consecutive scheduled departures with equal departure times. Although the calculation is consistent, these records may represent simultaneous scheduled trips and should not automatically be treated as duplicate errors. Further checks against the original GTFS data may be required to understand their operational meaning.

**Conclusion:** The headway calculation passed the verification check for Hyderabad. The existing data was retained without modification.

In [27]:
import pandas as pd

cities = ["ahmedabad", "chennai", "hyderabad", "mumbai", "pune"]

for city in cities:
    path = PROCESSED_DIR / f"{city}_trip_time_features.csv"
    df = pd.read_csv(path)

    print(f"\n--- {city.title()} ---")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    print("\nMissing values:")
    print(df.isna().sum().to_string())


--- Ahmedabad ---
Rows: 14323
Columns: ['city', 'trip_id', 'route_id', 'service_id', 'departure_time', 'departure_time_seconds', 'service_hour', 'clock_hour', 'minute', 'time_period']

Missing values:
city                      0
trip_id                   0
route_id                  0
service_id                0
departure_time            0
departure_time_seconds    0
service_hour              0
clock_hour                0
minute                    0
time_period               0

--- Chennai ---
Rows: 48435
Columns: ['city', 'trip_id', 'route_id', 'service_id', 'departure_time', 'departure_time_seconds', 'service_hour', 'clock_hour', 'minute', 'time_period']

Missing values:
city                      0
trip_id                   0
route_id                  0
service_id                0
departure_time            0
departure_time_seconds    0
service_hour              0
clock_hour                0
minute                    0
time_period               0

--- Hyderabad ---
Rows: 44892
Columns

In [28]:
import pandas as pd

cities = ["ahmedabad", "chennai", "hyderabad", "mumbai", "pune"]
results = []

for city in cities:
    path = PROCESSED_DIR / f"{city}_trip_time_features.csv"
    df = pd.read_csv(path)

    results.append({
        "City": city.title(),
        "Total rows": len(df),
        "Invalid seconds": int(
            (~df["departure_time_seconds"].between(0, 172799)).sum()
        ),
        "Invalid clock hours": int(
            (~df["clock_hour"].between(0, 23)).sum()
        ),
        "Invalid minutes": int(
            (~df["minute"].between(0, 59)).sum()
        ),
        "Invalid service hours": int(
            (df["service_hour"] < 0).sum()
        ),
        "Missing time periods": int(df["time_period"].isna().sum())
    })

display(pd.DataFrame(results))

,City,Total rows,Invalid seconds,Invalid clock hours,Invalid minutes,Invalid service hours,Missing time periods
0,Ahmedabad,14323,0,0,0,0,0
1,Chennai,48435,0,0,0,0,0
2,Hyderabad,44892,8,0,0,0,0
3,Mumbai,99080,0,0,0,0,0
4,Pune,10128,0,0,0,0,0


In [29]:
import pandas as pd

path = PROCESSED_DIR / "hyderabad_trip_time_features.csv"
df = pd.read_csv(path)

invalid = df[
    ~df["departure_time_seconds"].between(0, 172799)
]

print("Total rows:", len(df))
print("Records outside checked range:", len(invalid))

print(
    invalid[
        [
            "trip_id",
            "route_id",
            "service_id",
            "departure_time",
            "departure_time_seconds",
            "service_hour",
            "clock_hour",
            "minute",
            "time_period",
        ]
    ].to_string(index=False)
)

Total rows: 44892
Records outside checked range: 8
 trip_id route_id service_id departure_time  departure_time_seconds  service_hour  clock_hour  minute time_period
41854287     222P    MTWTFSS       56:45:00                  204300            56           8      45     Morning
41854288     222P    MTWTFSS       69:05:00                  248700            69          21       5       Night
41854289     222P    MTWTFSS       80:45:00                  290700            80           8      45     Morning
41854290     222P    MTWTFSS       86:05:00                  309900            86          14       5   Afternoon
41854444     222P    MTWTFSS       56:45:00                  204300            56           8      45     Morning
41854445     222P    MTWTFSS       69:05:00                  248700            69          21       5       Night
41854446     222P    MTWTFSS       80:45:00                  290700            80           8      45     Morning
41854447     222P    MTWTFSS       86

In [30]:
import pandas as pd

path = PROCESSED_DIR / "hyderabad_trip_time_features.csv"
df = pd.read_csv(path)

invalid = df[
    ~df["departure_time_seconds"].between(0, 172799)
].copy()

def time_to_seconds(value):
    hours, minutes, seconds = map(int, value.split(":"))
    return hours * 3600 + minutes * 60 + seconds

invalid["calculated_seconds"] = invalid["departure_time"].apply(
    time_to_seconds
)

invalid["matches"] = (
    invalid["departure_time_seconds"] == invalid["calculated_seconds"]
)

print("Extended-time records:", len(invalid))
print("Records with matching seconds:", int(invalid["matches"].sum()))
print("Records with mismatched seconds:", int((~invalid["matches"]).sum()))

print(
    invalid[
        ["departure_time", "departure_time_seconds",
         "calculated_seconds", "matches"]
    ].to_string(index=False)
)

Extended-time records: 8
Records with matching seconds: 8
Records with mismatched seconds: 0
departure_time  departure_time_seconds  calculated_seconds  matches
      56:45:00                  204300              204300     True
      69:05:00                  248700              248700     True
      80:45:00                  290700              290700     True
      86:05:00                  309900              309900     True
      56:45:00                  204300              204300     True
      69:05:00                  248700              248700     True
      80:45:00                  290700              290700     True
      86:05:00                  309900              309900     True


### Trip-Time Feature Validation – Results and Observations

The trip-time feature files for Ahmedabad, Chennai, Hyderabad, Mumbai, and Pune were checked for missing values and time-related inconsistencies. All five datasets contained complete values across the ten trip-time feature columns.

The clock-hour and minute values were within their expected ranges, and no negative service-hour values or missing time-period labels were found. Eight Hyderabad records fell outside the initial two-day seconds range because their departure times exceeded 48 hours.

These eight records were examined individually. Their stored departure-time seconds matched the seconds independently calculated from the original extended GTFS time strings, with zero mismatches. Therefore, they were retained as valid extended-time representations rather than removed or modified.

**Conclusion:** The trip-time features passed the missing-value and consistency checks performed. Extended GTFS times were preserved, and the initial two-day range was treated as a validation threshold rather than a universal limit on valid GTFS times.

In [31]:
import pandas as pd

cities = ["ahmedabad", "chennai", "hyderabad", "mumbai", "pune"]
results = []

for city in cities:
    path = PROCESSED_DIR / f"{city}_trip_time_features.csv"
    df = pd.read_csv(path)

    results.append({
        "City": city.title(),
        "Total rows": len(df),
        "Unique trip IDs": df["trip_id"].nunique(),
        "Duplicate trip IDs": int(df["trip_id"].duplicated().sum()),
        "Missing trip IDs": int(df["trip_id"].isna().sum())
    })

display(pd.DataFrame(results))

,City,Total rows,Unique trip IDs,Duplicate trip IDs,Missing trip IDs
0,Ahmedabad,14323,14323,0,0
1,Chennai,48435,48435,0,0
2,Hyderabad,44892,44892,0,0
3,Mumbai,99080,99080,0,0
4,Pune,10128,10128,0,0


In [32]:
from pathlib import Path

project_dir = Path.home() / "OneDrive" / "Desktop" / "RouteRanker"

print("Project folder exists:", project_dir.exists())
print("\nHeadway CSV files:")

headway_files = list(project_dir.rglob("*headway*.csv"))

if headway_files:
    for file in headway_files:
        print(file)
else:
    print("No headway CSV found. We need to check the output filenames.")


Project folder exists: True

Headway CSV files:
C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\data\processed\ahmedabad_trip_headway_features.csv
C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\data\processed\chennai_trip_headway_features.csv
C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\data\processed\headway_validation_report.csv
C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\data\processed\hyderabad_trip_headway_features.csv
C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\data\processed\mumbai_trip_headway_features.csv
C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\data\processed\pune_trip_headway_features.csv


In [33]:

import pandas as pd
from pathlib import Path

processed_dir = project_dir / "data" / "processed"

headway_files = sorted(processed_dir.glob("*_trip_headway_features.csv"))

for file in headway_files:
    df = pd.read_csv(file)

    print("\n" + "=" * 60)
    print("CITY:", file.name)
    print("Rows and columns:", df.shape)
    print("\nColumn names:")
    print(df.columns.tolist())
    print("\nFirst 5 rows:")
    display(df.head())
    print("\nMissing values:")
    display(df.isnull().sum().to_frame("Missing count"))



CITY: ahmedabad_trip_headway_features.csv
Rows and columns: (14323, 11)

Column names:
['city', 'trip_id', 'route_id', 'service_id', 'departure_time', 'departure_time_seconds', 'service_hour', 'clock_hour', 'minute', 'time_period', 'headway_minutes']

First 5 rows:


,city,trip_id,route_id,service_id,departure_time,departure_time_seconds,service_hour,clock_hour,minute,time_period,headway_minutes
0,ahmedabad,AMC_9347738,101-C,"1,2,3,4,5,6,7",06:00:00,21600,6,6,0,Morning,NaN
1,ahmedabad,AMC_9347819,101-C,"1,2,3,4,5,6,7",06:12:00,22320,6,6,12,Morning,12.0
2,ahmedabad,AMC_9347827,101-C,"1,2,3,4,5,6,7",06:24:00,23040,6,6,24,Morning,12.0
3,ahmedabad,AMC_9347860,101-C,"1,2,3,4,5,6,7",06:36:00,23760,6,6,36,Morning,12.0
4,ahmedabad,AMC_9347870,101-C,"1,2,3,4,5,6,7",06:46:00,24360,6,6,46,Morning,10.0



Missing values:


,Missing count
city,0
trip_id,0
route_id,0
service_id,0
departure_time,0
departure_time_seconds,0
service_hour,0
clock_hour,0
minute,0
time_period,0



CITY: chennai_trip_headway_features.csv
Rows and columns: (48435, 11)

Column names:
['city', 'trip_id', 'route_id', 'service_id', 'departure_time', 'departure_time_seconds', 'service_hour', 'clock_hour', 'minute', 'time_period', 'headway_minutes']

First 5 rows:


,city,trip_id,route_id,service_id,departure_time,departure_time_seconds,service_hour,clock_hour,minute,time_period,headway_minutes
0,chennai,CMRL-WD1021201,CMRL-102101,CMRL-WD,05:29:58,19798,5,5,29,Morning,NaN
1,chennai,CMRL-WD1025001,CMRL-102103,CMRL-WD,16:00:23,57623,16,16,0,Afternoon,NaN
2,chennai,CMRL-WD1020301,CMRL-102146,CMRL-WD,05:27:37,19657,5,5,27,Morning,NaN
3,chennai,CMRL-WD1091801,CMRL-109108,CMRL-WD,05:32:24,19944,5,5,32,Morning,NaN
4,chennai,CMRL-WD1090801,CMRL-109114,CMRL-WD,05:28:39,19719,5,5,28,Morning,NaN


... (10 outputs left out)


CITY: pune_trip_headway_features.csv
Rows and columns: (10128, 11)

Column names:
['city', 'trip_id', 'route_id', 'service_id', 'departure_time', 'departure_time_seconds', 'service_hour', 'clock_hour', 'minute', 'time_period', 'headway_minutes']

First 5 rows:


,city,trip_id,route_id,service_id,departure_time,departure_time_seconds,service_hour,clock_hour,minute,time_period,headway_minutes
0,pune,trip_7_0,100,WEEKDAY,04:45:00,17100,4,4,45,Night,NaN
1,pune,trip_7_1,100,WEEKDAY,05:00:00,18000,5,5,0,Morning,15.0
2,pune,trip_7_2,100,WEEKDAY,05:15:00,18900,5,5,15,Morning,15.0
3,pune,trip_7_3,100,WEEKDAY,05:30:00,19800,5,5,30,Morning,15.0
4,pune,trip_7_4,100,WEEKDAY,05:45:00,20700,5,5,45,Morning,15.0



Missing values:


,Missing count
city,0
trip_id,0
route_id,0
service_id,0
departure_time,0
departure_time_seconds,0
service_hour,0
clock_hour,0
minute,0
time_period,0


In [34]:

import numpy as np
import pandas as pd

group_cols = ["city", "route_id", "service_id"]

for file in sorted(processed_dir.glob("*_trip_headway_features.csv")):
    df = pd.read_csv(file)

    # Sort departures within each route and service group
    ordered = df.sort_values(
        group_cols + ["departure_time_seconds"]
    ).copy()

    # Recalculate the interval between consecutive departures
    ordered["expected_headway"] = (
        ordered.groupby(group_cols)["departure_time_seconds"]
        .diff() / 60
    )

    # Compare with the existing headway values
    existing = pd.to_numeric(
        ordered["headway_minutes"], errors="coerce"
    )
    expected = ordered["expected_headway"]

    matches = np.isclose(
        existing.to_numpy(dtype=float),
        expected.to_numpy(dtype=float),
        equal_nan=True
    )

    print("\n", "=" * 55)
    print("CITY:", file.stem)
    print("Total records:", len(ordered))
    print("Missing headways:", existing.isna().sum())
    print("Zero-minute headways:", (existing == 0).sum())
    print("Negative headways:", (existing < 0).sum())
    print("Matching calculated headways:", matches.sum())
    print("Mismatched headways:", (~matches).sum())

    if (~matches).any():
        print("\nFirst mismatched records:")
        display(
            ordered.loc[
                ~matches,
                group_cols + [
                    "trip_id",
                    "departure_time",
                    "headway_minutes",
                    "expected_headway"
                ]
            ].head(5)
        )



CITY: ahmedabad_trip_headway_features
Total records: 14323
Missing headways: 633
Zero-minute headways: 933
Negative headways: 0
Matching calculated headways: 14323
Mismatched headways: 0

CITY: chennai_trip_headway_features
Total records: 48435
Missing headways: 4732
Zero-minute headways: 1005
Negative headways: 0
Matching calculated headways: 48435
Mismatched headways: 0

CITY: hyderabad_trip_headway_features
Total records: 44892
Missing headways: 1031
Zero-minute headways: 13462
Negative headways: 0
Matching calculated headways: 44892
Mismatched headways: 0

CITY: mumbai_trip_headway_features
Total records: 99080
Missing headways: 612
Zero-minute headways: 9257
Negative headways: 0
Matching calculated headways: 99080
Mismatched headways: 0

CITY: pune_trip_headway_features
Total records: 10128
Missing headways: 315
Zero-minute headways: 1507
Negative headways: 0
Matching calculated headways: 10128
Mismatched headways: 0


In [35]:

for city in ["ahmedabad", "chennai", "hyderabad", "mumbai", "pune"]:
    file = processed_dir / f"{city}_trip_headway_features.csv"
    df = pd.read_csv(file)

    print("\n" + "=" * 60)
    print("CITY:", city)

    zero = df[df["headway_minutes"].eq(0)]

    print("Zero-headway records:", len(zero))

    display(
        zero[
            [
                "route_id",
                "service_id",
                "trip_id",
                "departure_time",
                "departure_time_seconds",
                "headway_minutes"
            ]
        ].head(10)
    )



CITY: ahmedabad
Zero-headway records: 933


,route_id,service_id,trip_id,departure_time,departure_time_seconds,headway_minutes
173,103-2,"1,2,3,4,5,6,7",AMC_9348986,21:20:00,76800,0.0
189,105,"1,2,3,4,5,6,7",AMC_9349187,06:05:00,21900,0.0
194,105,"1,2,3,4,5,6,7",AMC_9349229,06:35:00,23700,0.0
196,105,"1,2,3,4,5,6,7",AMC_9349160,06:45:00,24300,0.0
198,105,"1,2,3,4,5,6,7",AMC_9349174,06:55:00,24900,0.0
200,105,"1,2,3,4,5,6,7",AMC_9349188,07:05:00,25500,0.0
203,105,"1,2,3,4,5,6,7",AMC_9349202,07:15:00,26100,0.0
205,105,"1,2,3,4,5,6,7",AMC_9349216,07:25:00,26700,0.0
216,105,"1,2,3,4,5,6,7",AMC_9349103,08:15:00,29700,0.0
219,105,"1,2,3,4,5,6,7",AMC_9349189,08:25:00,30300,0.0



CITY: chennai
Zero-headway records: 1005


,route_id,service_id,trip_id,departure_time,departure_time_seconds,headway_minutes
1196,MTC-10913,MTC-Regular,MTC-64563,05:25:00,19500,0.0
1236,MTC-10913,MTC-Regular,MTC-64575,19:25:00,69900,0.0
1245,MTC-10914,MTC-Regular,MTC-64564,06:30:00,23400,0.0
1718,MTC-11082,MTC-Regular,MTC-16341,22:10:00,79800,0.0
2519,MTC-11464,MTC-Regular,MTC-27392,06:52:00,24720,0.0
2536,MTC-11464,MTC-Regular,MTC-27394,09:12:00,33120,0.0
2551,MTC-11464,MTC-Regular,MTC-27396,11:32:00,41520,0.0
2573,MTC-11464,MTC-Regular,MTC-27401,16:22:00,58920,0.0
2589,MTC-11464,MTC-Regular,MTC-27403,18:42:00,67320,0.0
2613,MTC-11465,MTC-Regular,MTC-27316,21:20:00,76800,0.0



CITY: hyderabad
Zero-headway records: 13462


,route_id,service_id,trip_id,departure_time,departure_time_seconds,headway_minutes
3,1,MTWTFSS,41748569,04:10:00,15000,0.0
8,1,MTWTFSS,41735584,05:20:00,19200,0.0
11,1,MTWTFSS,41748571,05:30:00,19800,0.0
16,1,MTWTFSS,41735585,06:00:00,21600,0.0
19,1,MTWTFSS,41748294,06:20:00,22800,0.0
29,1,MTWTFSS,41748295,07:00:00,25200,0.0
60,1,MTWTFSS,41748300,10:30:00,37800,0.0
72,1,MTWTFSS,41751526,11:30:00,41400,0.0
81,1,MTWTFSS,41756636,12:00:00,43200,0.0
84,1,MTWTFSS,41729880,12:10:00,43800,0.0



CITY: mumbai
Zero-headway records: 9257


,route_id,service_id,trip_id,departure_time,departure_time_seconds,headway_minutes
3,1-2,"1,2,3,4,5,6,7",tmt_1_1_56,00:15:00,900,0.0
7,1-2,"1,2,3,4,5,6,7",tmt_1_1_58,00:35:00,2100,0.0
13,1-2,"1,2,3,4,5,6,7",tmt_1_1_61,01:00:00,3600,0.0
17,1-2,"1,2,3,4,5,6,7",tmt_1_1_63,01:20:00,4800,0.0
45,1-2,"1,2,3,4,5,6,7",tmt_1_1_13,11:20:00,40800,0.0
49,1-2,"1,2,3,4,5,6,7",tmt_1_1_15,11:40:00,42000,0.0
52,1-2,"1,2,3,4,5,6,7",tmt_1_1_16,12:00:00,43200,0.0
54,1-2,"1,2,3,4,5,6,7",tmt_1_1_18,12:05:00,43500,0.0
56,1-2,"1,2,3,4,5,6,7",tmt_1_1_19,12:20:00,44400,0.0
59,1-2,"1,2,3,4,5,6,7",tmt_1_1_20,12:40:00,45600,0.0



CITY: pune
Zero-headway records: 1507


,route_id,service_id,trip_id,departure_time,departure_time_seconds,headway_minutes
8,100,WEEKDAY,trip_7_6,06:15:00,22500,0.0
10,100,WEEKDAY,trip_7_7,06:30:00,23400,0.0
12,100,WEEKDAY,trip_7_8,06:45:00,24300,0.0
29,100,WEEKDAY,trip_7_16,08:50:00,31800,0.0
34,100,WEEKDAY,trip_7_18,09:20:00,33600,0.0
36,100,WEEKDAY,trip_7_19,09:35:00,34500,0.0
38,100,WEEKDAY,trip_7_20,09:50:00,35400,0.0
40,100,WEEKDAY,trip_7_21,10:05:00,36300,0.0
58,100,WEEKDAY,trip_7_30,12:40:00,45600,0.0
144,11,WEEKDAY,trip_69_7,09:10:00,33000,0.0


In [36]:

for city in ["ahmedabad", "chennai", "hyderabad", "mumbai", "pune"]:
    file = processed_dir / f"{city}_trip_headway_features.csv"
    df = pd.read_csv(file)

    # Count trips with the same route, service and departure time
    duplicate_mask = df.duplicated(
        subset=["route_id", "service_id", "departure_time_seconds"],
        keep=False
    )

    duplicates = df.loc[duplicate_mask].sort_values(
        ["route_id", "service_id", "departure_time_seconds"]
    )

    print("\n" + "=" * 55)
    print("CITY:", city)
    print("Rows sharing a departure time:", len(duplicates))

    display(
        duplicates[
            [
                "route_id",
                "service_id",
                "trip_id",
                "departure_time",
                "headway_minutes"
            ]
        ].head(10)
    )



CITY: ahmedabad
Rows sharing a departure time: 1866


,route_id,service_id,trip_id,departure_time,headway_minutes
172,103-2,"1,2,3,4,5,6,7",AMC_9348974,21:20:00,55.0
173,103-2,"1,2,3,4,5,6,7",AMC_9348986,21:20:00,0.0
188,105,"1,2,3,4,5,6,7",AMC_9348992,06:05:00,10.0
189,105,"1,2,3,4,5,6,7",AMC_9349187,06:05:00,0.0
193,105,"1,2,3,4,5,6,7",AMC_9349146,06:35:00,10.0
194,105,"1,2,3,4,5,6,7",AMC_9349229,06:35:00,0.0
195,105,"1,2,3,4,5,6,7",AMC_9349005,06:45:00,10.0
196,105,"1,2,3,4,5,6,7",AMC_9349160,06:45:00,0.0
197,105,"1,2,3,4,5,6,7",AMC_9349019,06:55:00,10.0
198,105,"1,2,3,4,5,6,7",AMC_9349174,06:55:00,0.0



CITY: chennai
Rows sharing a departure time: 1984


,route_id,service_id,trip_id,departure_time,headway_minutes
1195,MTC-10913,MTC-Regular,MTC-62101,05:25:00,25.0
1196,MTC-10913,MTC-Regular,MTC-64563,05:25:00,0.0
1235,MTC-10913,MTC-Regular,MTC-62113,19:25:00,35.0
1236,MTC-10913,MTC-Regular,MTC-64575,19:25:00,0.0
1244,MTC-10914,MTC-Regular,MTC-62102,06:30:00,25.0
1245,MTC-10914,MTC-Regular,MTC-64564,06:30:00,0.0
1717,MTC-11082,MTC-Regular,MTC-16315,22:10:00,15.0
1718,MTC-11082,MTC-Regular,MTC-16341,22:10:00,0.0
2518,MTC-11464,MTC-Regular,MTC-27320,06:52:00,14.0
2519,MTC-11464,MTC-Regular,MTC-27392,06:52:00,0.0



CITY: hyderabad
Rows sharing a departure time: 21000


,route_id,service_id,trip_id,departure_time,headway_minutes
2,1,MTWTFSS,41729872,04:10:00,5.0
3,1,MTWTFSS,41748569,04:10:00,0.0
7,1,MTWTFSS,41729888,05:20:00,10.0
8,1,MTWTFSS,41735584,05:20:00,0.0
10,1,MTWTFSS,41729762,05:30:00,3.0
11,1,MTWTFSS,41748571,05:30:00,0.0
15,1,MTWTFSS,1109118,06:00:00,10.0
16,1,MTWTFSS,41735585,06:00:00,0.0
18,1,MTWTFSS,41736676,06:20:00,10.0
19,1,MTWTFSS,41748294,06:20:00,0.0



CITY: mumbai
Rows sharing a departure time: 18491


,route_id,service_id,trip_id,departure_time,headway_minutes
2,1-2,"1,2,3,4,5,6,7",tmt_1_0_52,00:15:00,5.0
3,1-2,"1,2,3,4,5,6,7",tmt_1_1_56,00:15:00,0.0
6,1-2,"1,2,3,4,5,6,7",tmt_1_0_54,00:35:00,5.0
7,1-2,"1,2,3,4,5,6,7",tmt_1_1_58,00:35:00,0.0
12,1-2,"1,2,3,4,5,6,7",tmt_1_0_57,01:00:00,5.0
13,1-2,"1,2,3,4,5,6,7",tmt_1_1_61,01:00:00,0.0
16,1-2,"1,2,3,4,5,6,7",tmt_1_0_59,01:20:00,5.0
17,1-2,"1,2,3,4,5,6,7",tmt_1_1_63,01:20:00,0.0
44,1-2,"1,2,3,4,5,6,7",tmt_1_0_14,11:20:00,5.0
45,1-2,"1,2,3,4,5,6,7",tmt_1_1_13,11:20:00,0.0



CITY: pune
Rows sharing a departure time: 3014


,route_id,service_id,trip_id,departure_time,headway_minutes
7,100,WEEKDAY,trip_6_1,06:15:00,15.0
8,100,WEEKDAY,trip_7_6,06:15:00,0.0
9,100,WEEKDAY,trip_6_2,06:30:00,15.0
10,100,WEEKDAY,trip_7_7,06:30:00,0.0
11,100,WEEKDAY,trip_6_3,06:45:00,15.0
12,100,WEEKDAY,trip_7_8,06:45:00,0.0
28,100,WEEKDAY,trip_6_12,08:50:00,10.0
29,100,WEEKDAY,trip_7_16,08:50:00,0.0
33,100,WEEKDAY,trip_6_15,09:20:00,10.0
34,100,WEEKDAY,trip_7_18,09:20:00,0.0


In [37]:

import pandas as pd
from pathlib import Path

summary = []

for city in ["ahmedabad", "chennai", "hyderabad", "mumbai", "pune"]:
    file = processed_dir / f"{city}_trip_headway_features.csv"
    df = pd.read_csv(file)

    group_cols = ["route_id", "service_id"]

    # Keep distinct departure times for interval analysis
    unique_departures = (
        df[group_cols + ["departure_time_seconds"]]
        .drop_duplicates()
        .sort_values(group_cols + ["departure_time_seconds"])
        .copy()
    )

    # Calculate intervals between distinct departure times
    unique_departures["distinct_headway_minutes"] = (
        unique_departures
        .groupby(group_cols)["departure_time_seconds"]
        .diff() / 60
    )

    summary.append({
        "city": city,
        "total_trip_records": len(df),
        "distinct_departure_records": len(unique_departures),
        "simultaneous_departure_records":
            len(df) - len(unique_departures),
        "missing_distinct_headways":
            unique_departures["distinct_headway_minutes"].isna().sum(),
        "zero_distinct_headways":
            unique_departures["distinct_headway_minutes"].eq(0).sum(),
        "negative_distinct_headways":
            unique_departures["distinct_headway_minutes"].lt(0).sum()
    })

summary_df = pd.DataFrame(summary)
display(summary_df)


,city,total_trip_records,distinct_departure_records,simultaneous_departure_records,missing_distinct_headways,zero_distinct_headways,negative_distinct_headways
0,ahmedabad,14323,13390,933,633,0,0
1,chennai,48435,47430,1005,4732,0,0
2,hyderabad,44892,31430,13462,1031,0,0
3,mumbai,99080,89823,9257,612,0,0
4,pune,10128,8621,1507,315,0,0


In [38]:
import pandas as pd

validation_results = []

for city in ["ahmedabad", "chennai", "hyderabad", "mumbai", "pune"]:
    file = processed_dir / f"{city}_trip_headway_features.csv"
    df = pd.read_csv(file)

    group_cols = ["route_id", "service_id"]

    unique_departures = (
        df[group_cols + ["departure_time_seconds"]]
        .drop_duplicates()
        .sort_values(group_cols + ["departure_time_seconds"])
        .copy()
    )

    unique_departures["expected_headway_minutes"] = (
        unique_departures
        .groupby(group_cols)["departure_time_seconds"]
        .diff() / 60
    )

    expected = unique_departures[
        group_cols + [
            "departure_time_seconds",
            "expected_headway_minutes"
        ]
    ]

    checked = df.merge(
        expected,
        on=group_cols + ["departure_time_seconds"],
        how="left"
    )

    validation_results.append({
        "city": city,
        "trip_records": len(df),
        "missing_original_headways":
            df["headway_minutes"].isna().sum(),
        "missing_expected_headways":
            checked["expected_headway_minutes"].isna().sum(),
        "zero_expected_headways":
            checked["expected_headway_minutes"].eq(0).sum(),
        "negative_expected_headways":
            checked["expected_headway_minutes"].lt(0).sum()
    })

validation_df = pd.DataFrame(validation_results)
display(validation_df)

,city,trip_records,missing_original_headways,missing_expected_headways,zero_expected_headways,negative_expected_headways
0,ahmedabad,14323,633,640,0,0
1,chennai,48435,4732,4751,0,0
2,hyderabad,44892,1031,1128,0,0
3,mumbai,99080,612,629,0,0
4,pune,10128,315,321,0,0


In [39]:
import pandas as pd

city = "ahmedabad"

file = processed_dir / f"{city}_trip_headway_features.csv"
df = pd.read_csv(file)

print("Total records:", len(df))
print("Missing original headways:", df["headway_minutes"].isna().sum())

display(
    df.loc[
        df["headway_minutes"].isna(),
        [
            "route_id",
            "service_id",
            "trip_id",
            "departure_time",
            "departure_time_seconds",
            "headway_minutes"
        ]
    ].head(20)
)

Total records: 14323
Missing original headways: 633


,route_id,service_id,trip_id,departure_time,departure_time_seconds,headway_minutes
0,101-C,"1,2,3,4,5,6,7",AMC_9347738,06:00:00,21600,NaN
96,101-C-PR,"1,2,3,4,5,6,7",AMC_9347776,06:38:00,23880,NaN
100,101-C-RS,"1,2,3,4,5,6,7",AMC_9347780,12:41:00,45660,NaN
103,101S-RM,"1,2,3,4,5,6,7",AMC_9347866,21:39:00,77940,NaN
109,101U-NR,"1,2,3,4,5,6,7",AMC_9347859,06:00:00,21600,NaN
115,101sh,"1,2,3,4,5,6,7",AMC_9348909,06:20:00,22800,NaN
125,102-1,"1,2,3,4,5,6,7",AMC_9348924,12:30:00,45000,NaN
131,102-2,"1,2,3,4,5,6,7",AMC_9348921,06:10:00,22200,NaN
133,102-3,"1,2,3,4,5,6,7",AMC_9348922,07:40:00,27600,NaN
141,103-1,"1,2,3,4,5,6,7",AMC_9348942,06:35:00,23700,NaN


In [40]:
route_id = "101-C"

route_df = df[df["route_id"] == route_id].copy()

route_df = route_df.sort_values("departure_time_seconds")

display(
    route_df[
        [
            "route_id",
            "service_id",
            "trip_id",
            "departure_time",
            "departure_time_seconds",
            "headway_minutes"
        ]
    ].head(15)
)

,route_id,service_id,trip_id,departure_time,departure_time_seconds,headway_minutes
0,101-C,"1,2,3,4,5,6,7",AMC_9347738,06:00:00,21600,NaN
1,101-C,"1,2,3,4,5,6,7",AMC_9347819,06:12:00,22320,12.0
2,101-C,"1,2,3,4,5,6,7",AMC_9347827,06:24:00,23040,12.0
3,101-C,"1,2,3,4,5,6,7",AMC_9347860,06:36:00,23760,12.0
4,101-C,"1,2,3,4,5,6,7",AMC_9347870,06:46:00,24360,10.0
5,101-C,"1,2,3,4,5,6,7",AMC_9347777,06:50:00,24600,4.0
6,101-C,"1,2,3,4,5,6,7",AMC_9347880,06:56:00,24960,6.0
7,101-C,"1,2,3,4,5,6,7",AMC_9347747,07:06:00,25560,10.0
8,101-C,"1,2,3,4,5,6,7",AMC_9347790,07:10:00,25800,4.0
9,101-C,"1,2,3,4,5,6,7",AMC_9347757,07:14:00,26040,4.0


In [41]:
import pandas as pd

cities = ["ahmedabad", "chennai", "hyderabad", "mumbai", "pune"]
results = []

for city in cities:
    file = processed_dir / f"{city}_trip_headway_features.csv"
    data = pd.read_csv(file)

    data = data.sort_values(
        ["route_id", "service_id", "departure_time_seconds"]
    ).copy()

    expected = (
        data.groupby(["route_id", "service_id"])["departure_time_seconds"]
        .diff()
        .div(60)
    )

    actual = data["headway_minutes"]

    matches = (
        (actual.isna() & expected.isna())
        | (actual.notna() & expected.notna() & actual.eq(expected))
    )

    results.append({
        "city": city,
        "total_records": len(data),
        "matching_headways": int(matches.sum()),
        "mismatched_headways": int((~matches).sum()),
        "missing_headways": int(actual.isna().sum())
    })

display(pd.DataFrame(results))

,city,total_records,matching_headways,mismatched_headways,missing_headways
0,ahmedabad,14323,14323,0,633
1,chennai,48435,48435,0,4732
2,hyderabad,44892,44892,0,1031
3,mumbai,99080,99080,0,612
4,pune,10128,10128,0,315


In [42]:
from pathlib import Path
import pandas as pd

validation_report = pd.DataFrame([
    {"city": "ahmedabad", "total_records": 14323,
     "matching_headways": 14323, "mismatched_headways": 0,
     "missing_headways": 633},
    {"city": "chennai", "total_records": 48435,
     "matching_headways": 48435, "mismatched_headways": 0,
     "missing_headways": 4732},
    {"city": "hyderabad", "total_records": 44892,
     "matching_headways": 44892, "mismatched_headways": 0,
     "missing_headways": 1031},
    {"city": "mumbai", "total_records": 99080,
     "matching_headways": 99080, "mismatched_headways": 0,
     "missing_headways": 612},
    {"city": "pune", "total_records": 10128,
     "matching_headways": 10128, "mismatched_headways": 0,
     "missing_headways": 315}
])

report_path = processed_dir / "headway_validation_report.csv"
validation_report.to_csv(report_path, index=False)

print("Validation report saved to:", report_path)
display(validation_report)

Validation report saved to: C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\data\processed\headway_validation_report.csv


,city,total_records,matching_headways,mismatched_headways,missing_headways
0,ahmedabad,14323,14323,0,633
1,chennai,48435,48435,0,4732
2,hyderabad,44892,44892,0,1031
3,mumbai,99080,99080,0,612
4,pune,10128,10128,0,315


In [43]:

import pandas as pd

cities = ["ahmedabad", "chennai", "hyderabad", "mumbai", "pune"]

for city in cities:
    file = processed_dir / f"{city}_trip_headway_features.csv"
    data = pd.read_csv(file)

    print(f"\n{'=' * 50}")
    print(f"CITY: {city.upper()}")
    print("Rows:", len(data))
    print("Columns:", len(data.columns))
    print("\nColumn names:")
    print(data.columns.tolist())

    print("\nMissing values:")
    display(
        data.isna().sum()
        .rename("missing_count")
        .to_frame()
        .query("missing_count > 0")
    )



CITY: AHMEDABAD
Rows: 14323
Columns: 11

Column names:
['city', 'trip_id', 'route_id', 'service_id', 'departure_time', 'departure_time_seconds', 'service_hour', 'clock_hour', 'minute', 'time_period', 'headway_minutes']

Missing values:


,missing_count
headway_minutes,633



CITY: CHENNAI
Rows: 48435
Columns: 11

Column names:
['city', 'trip_id', 'route_id', 'service_id', 'departure_time', 'departure_time_seconds', 'service_hour', 'clock_hour', 'minute', 'time_period', 'headway_minutes']

Missing values:


,missing_count
headway_minutes,4732



CITY: HYDERABAD
Rows: 44892
Columns: 11

Column names:
['city', 'trip_id', 'route_id', 'service_id', 'departure_time', 'departure_time_seconds', 'service_hour', 'clock_hour', 'minute', 'time_period', 'headway_minutes']

Missing values:


,missing_count
headway_minutes,1031



CITY: MUMBAI
Rows: 99080
Columns: 11

Column names:
['city', 'trip_id', 'route_id', 'service_id', 'departure_time', 'departure_time_seconds', 'service_hour', 'clock_hour', 'minute', 'time_period', 'headway_minutes']

Missing values:


,missing_count
headway_minutes,612



CITY: PUNE
Rows: 10128
Columns: 11

Column names:
['city', 'trip_id', 'route_id', 'service_id', 'departure_time', 'departure_time_seconds', 'service_hour', 'clock_hour', 'minute', 'time_period', 'headway_minutes']

Missing values:


,missing_count
headway_minutes,315


In [44]:

import pandas as pd

cities = ["ahmedabad", "chennai", "hyderabad", "mumbai", "pune"]

for city in cities:
    file = processed_dir / f"{city}_trip_headway_features.csv"
    data = pd.read_csv(file)

    print(f"\n{'=' * 45}")
    print(f"CITY: {city.upper()}")

    print("service_hour range:",
          data["service_hour"].min(),
          "to",
          data["service_hour"].max())

    print("clock_hour range:",
          data["clock_hour"].min(),
          "to",
          data["clock_hour"].max())

    print("minute range:",
          data["minute"].min(),
          "to",
          data["minute"].max())

    print("\nTime period counts:")
    display(data["time_period"].value_counts(dropna=False).to_frame("count"))

    print("\nInvalid time-feature values:")
    print("Invalid clock_hour:",
          (~data["clock_hour"].between(0, 23)).sum())
    print("Invalid minute:",
          (~data["minute"].between(0, 59)).sum())



CITY: AHMEDABAD
service_hour range: 5 to 23
clock_hour range: 5 to 23
minute range: 0 to 59

Time period counts:


,count
time_period,
Morning,5812
Afternoon,4207
Evening,3323
Night,981



Invalid time-feature values:
Invalid clock_hour: 0
Invalid minute: 0

CITY: CHENNAI
service_hour range: 0 to 23
clock_hour range: 0 to 23
minute range: 0 to 59

Time period counts:


,count
time_period,
Morning,19003
Afternoon,14717
Evening,10588
Night,4127



Invalid time-feature values:
Invalid clock_hour: 0
Invalid minute: 0

CITY: HYDERABAD
service_hour range: 1 to 86
clock_hour range: 0 to 23
minute range: 0 to 59

Time period counts:


,count
time_period,
Morning,18435
Afternoon,13769
Evening,10401
Night,2287



Invalid time-feature values:
Invalid clock_hour: 0
Invalid minute: 0

CITY: MUMBAI
service_hour range: 0 to 23
clock_hour range: 0 to 23
minute range: 0 to 59

Time period counts:


,count
time_period,
Morning,38043
Afternoon,25447
Evening,21958
Night,13632



Invalid time-feature values:
Invalid clock_hour: 0
Invalid minute: 0

CITY: PUNE
service_hour range: 0 to 23
clock_hour range: 0 to 23
minute range: 0 to 58

Time period counts:


,count
time_period,
Morning,6917
Afternoon,2870
Evening,215
Night,126



Invalid time-feature values:
Invalid clock_hour: 0
Invalid minute: 0


In [45]:

import pandas as pd

file = processed_dir / "hyderabad_trip_headway_features.csv"
hyd = pd.read_csv(file)

print("Records with service_hour >= 24:")
display(
    hyd.loc[
        hyd["service_hour"] >= 24,
        [
            "trip_id",
            "route_id",
            "departure_time",
            "departure_time_seconds",
            "service_hour",
            "clock_hour",
            "minute",
            "time_period"
        ]
    ]
    .sort_values("departure_time_seconds", ascending=False)
    .head(20)
)

print("\nMaximum departure time:")
display(
    hyd.nlargest(5, "departure_time_seconds")[
        ["trip_id", "route_id", "departure_time",
         "departure_time_seconds", "service_hour", "clock_hour"]
    ]
)


Records with service_hour >= 24:


,trip_id,route_id,departure_time,departure_time_seconds,service_hour,clock_hour,minute,time_period
20331,41854290,222P,86:05:00,309900,86,14,5,Afternoon
20332,41854447,222P,86:05:00,309900,86,14,5,Afternoon
20330,41854446,222P,80:45:00,290700,80,8,45,Morning
20329,41854289,222P,80:45:00,290700,80,8,45,Morning
20328,41854445,222P,69:05:00,248700,69,21,5,Night
20327,41854288,222P,69:05:00,248700,69,21,5,Night
20325,41854287,222P,56:45:00,204300,56,8,45,Morning
20326,41854444,222P,56:45:00,204300,56,8,45,Morning
34614,41774998,459,45:45:00,164700,45,21,45,Night
3282,41751714,10H,45:30:00,163800,45,21,30,Night



Maximum departure time:


,trip_id,route_id,departure_time,departure_time_seconds,service_hour,clock_hour
20331,41854290,222P,86:05:00,309900,86,14
20332,41854447,222P,86:05:00,309900,86,14
20329,41854289,222P,80:45:00,290700,80,8
20330,41854446,222P,80:45:00,290700,80,8
20327,41854288,222P,69:05:00,248700,69,21


In [46]:

from pathlib import Path

project_dir = Path(r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker")

print("Possible Hyderabad source files:")

for path in project_dir.rglob("stop_times.txt"):
    if "hyderabad" in str(path).lower():
        print(path)


Possible Hyderabad source files:
C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\data\raw\hyderabad\stop_times.txt


In [47]:

import pandas as pd
from pathlib import Path

source_file = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\data\raw\hyderabad\stop_times.txt"
)

# Inspect the source records for the two unusual trips
target_trips = ["41854290", "41854447"]

source = pd.read_csv(
    source_file,
    sep=",",
    dtype=str,
    usecols=lambda col: col in [
        "trip_id",
        "arrival_time",
        "departure_time",
        "stop_sequence"
    ]
)

source["trip_id"] = source["trip_id"].astype(str)

result = source[source["trip_id"].isin(target_trips)].copy()

print("Matching source records:", len(result))
display(
    result.sort_values(["trip_id", "stop_sequence"])
    .tail(20)
)


Matching source records: 8


,trip_id,stop_sequence,departure_time,arrival_time
1081737,41854290,1,86:05:00,86:05:00
1081738,41854290,2,86:08:59,86:08:59
1081739,41854290,3,86:15:48,86:15:48
1081740,41854290,4,86:25:48,86:25:48
1081733,41854447,1,86:05:00,86:05:00
1081734,41854447,2,86:08:59,86:08:59
1081735,41854447,3,86:15:48,86:15:48
1081736,41854447,4,86:25:48,86:25:48


In [48]:

import pandas as pd
from pathlib import Path

source_file = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\data\raw\hyderabad\stop_times.txt"
)

times = pd.read_csv(
    source_file,
    usecols=["trip_id", "departure_time"],
    dtype=str
)

# Parse GTFS HH:MM:SS, including hours above 23
parts = times["departure_time"].str.extract(
    r"^(\d+):([0-5]\d):([0-5]\d)$"
)

times["departure_hour"] = pd.to_numeric(parts[0], errors="coerce")

print("Total stop-time records:", len(times))
print("Unparseable departure times:", times["departure_hour"].isna().sum())
print("Records with departure hour >= 24:",
      (times["departure_hour"] >= 24).sum())
print("Records with departure hour >= 48:",
      (times["departure_hour"] >= 48).sum())
print("Records with departure hour >= 72:",
      (times["departure_hour"] >= 72).sum())

print("\nHighest departure hours:")
display(
    times["departure_hour"]
    .value_counts()
    .sort_index()
    .tail(15)
    .rename_axis("departure_hour")
    .to_frame("record_count")
)


Total stop-time records: 1132807
Unparseable departure times: 0
Records with departure hour >= 24: 327387
Records with departure hour >= 48: 24962
Records with departure hour >= 72: 3421

Highest departure hours:


,record_count
departure_hour,
229,1
233,1
236,1
240,1
243,1
247,1
254,1
261,1
268,1


In [49]:

import pandas as pd
from pathlib import Path

project_dir = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker"
)

stop_file = project_dir / "data" / "raw" / "hyderabad" / "stop_times.txt"
trips_file = project_dir / "data" / "raw" / "hyderabad" / "trips.txt"

st = pd.read_csv(
    stop_file,
    usecols=["trip_id", "stop_sequence", "arrival_time", "departure_time"],
    dtype=str
)

trips = pd.read_csv(
    trips_file,
    dtype=str
)

# Extract hour values without converting extended GTFS times to clock time
st["departure_hour"] = pd.to_numeric(
    st["departure_time"].str.extract(r"^(\d+):")[0],
    errors="coerce"
)

extreme = st[st["departure_hour"] >= 200].copy()

print("Records with departure hour >= 200:", len(extreme))
print("Distinct affected trips:", extreme["trip_id"].nunique())

print("\nExtreme departure-time records:")
display(
    extreme.sort_values("departure_hour", ascending=False).head(15)
)

affected_ids = extreme["trip_id"].drop_duplicates().head(5)

print("\nAll stop times for up to five affected trips:")
display(
    st[st["trip_id"].isin(affected_ids)]
    .sort_values(["trip_id", "stop_sequence"])
)

print("\nTrip metadata:")
display(
    trips[trips["trip_id"].isin(affected_ids)]
)


Records with departure hour >= 200: 23
Distinct affected trips: 2

Extreme departure-time records:


,trip_id,stop_sequence,departure_time,arrival_time,departure_hour
299208,41738902,87,311:57:58,311:57:58,311
299207,41738902,86,304:32:58,304:32:58,304
299206,41738902,85,297:12:38,297:12:38,297
299205,41738902,84,289:56:07,289:56:07,289
299204,41738902,83,282:42:03,282:42:03,282
299203,41738902,82,275:30:34,275:30:34,275
299202,41738902,81,268:21:39,268:21:39,268
299201,41738902,80,261:15:48,261:15:48,261
299200,41738902,79,254:11:34,254:11:34,254
299199,41738902,78,247:10:35,247:10:35,247



All stop times for up to five affected trips:


,trip_id,stop_sequence,departure_time,arrival_time,departure_hour
287324,41738901,1,06:25:00,06:25:00,6
287333,41738901,10,09:00:15,09:00:15,9
287334,41738901,11,09:38:10,09:38:10,9
287335,41738901,12,10:22:40,10:22:40,10
287336,41738901,13,11:11:49,11:11:49,11
...,...,...,...,...,...
299205,41738902,84,289:56:07,289:56:07,289
299206,41738902,85,297:12:38,297:12:38,297
299207,41738902,86,304:32:58,304:32:58,304
299208,41738902,87,311:57:58,311:57:58,311



Trip metadata:


,route_id,trip_id,service_id,direction_id,trip_short_name
10402,300,41738901,MTWTFSS,1,300-UPPAL DEPOT-TO-MEHDIPATNAM-UPL
10739,300,41738902,MTWTFSS,0,300-MEHDIPATNAM-TO-JBS-UPL


In [50]:

import pandas as pd
from pathlib import Path

project_dir = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker"
)

stop_file = project_dir / "data" / "raw" / "hyderabad" / "stop_times.txt"

st = pd.read_csv(
    stop_file,
    usecols=[
        "trip_id",
        "stop_sequence",
        "arrival_time",
        "departure_time"
    ],
    dtype=str
)

target_trips = ["41738901", "41738902"]
check = st[st["trip_id"].isin(target_trips)].copy()

def to_seconds(value):
    parts = str(value).split(":")
    if len(parts) != 3:
        return None
    return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])

check["departure_seconds"] = check["departure_time"].map(to_seconds)
check["stop_sequence_num"] = pd.to_numeric(
    check["stop_sequence"], errors="coerce"
)

check = check.sort_values(["trip_id", "stop_sequence_num"])

check["gap_from_previous_minutes"] = (
    check.groupby("trip_id")["departure_seconds"].diff() / 60
)

for trip in target_trips:
    print(f"\nTrip {trip}:")
    display(
        check.loc[
            check["trip_id"] == trip,
            [
                "stop_sequence",
                "arrival_time",
                "departure_time",
                "gap_from_previous_minutes"
            ]
        ].head(15)
    )

    print("Largest gaps between consecutive stops:")
    display(
        check.loc[check["trip_id"] == trip]
        .nlargest(5, "gap_from_previous_minutes")[
            [
                "stop_sequence",
                "departure_time",
                "gap_from_previous_minutes"
            ]
        ]
    )



Trip 41738901:


,stop_sequence,arrival_time,departure_time,gap_from_previous_minutes
287324,1,06:25:00,06:25:00,NaN
287325,2,06:26:39,06:26:39,1.650000
287326,3,06:31:43,06:31:43,5.066667
287327,4,06:42:45,06:42:45,11.033333
287328,5,06:56:43,06:56:43,13.966667
287329,6,07:13:26,07:13:26,16.716667
287330,7,07:31:04,07:31:04,17.633333
287331,8,07:54:01,07:54:01,22.950000
287332,9,08:25:03,08:25:03,31.033333
287333,10,09:00:15,09:00:15,35.200000


Largest gaps between consecutive stops:


,stop_sequence,departure_time,gap_from_previous_minutes
287396,73,243:13:38,430.000000
287395,72,236:03:38,419.883333
287394,71,229:03:45,405.983333
287393,70,222:17:46,393.566667
287392,69,215:44:12,387.883333



Trip 41738902:


,stop_sequence,arrival_time,departure_time,gap_from_previous_minutes
299122,1,08:05:00,08:05:00,NaN
299123,2,08:08:24,08:08:24,3.400000
299124,3,08:14:34,08:14:34,6.166667
299125,4,08:22:28,08:22:28,7.900000
299126,5,08:35:05,08:35:05,12.616667
299127,6,08:50:10,08:50:10,15.083333
299128,7,09:09:05,09:09:05,18.916667
299129,8,09:32:34,09:32:34,23.483333
299130,9,10:02:25,10:02:25,29.850000
299131,10,10:34:33,10:34:33,32.133333


Largest gaps between consecutive stops:


,stop_sequence,departure_time,gap_from_previous_minutes
299208,87,311:57:58,445.000000
299207,86,304:32:58,440.333333
299206,85,297:12:38,436.516667
299205,84,289:56:07,434.066667
299204,83,282:42:03,431.483333


In [51]:

import pandas as pd
from pathlib import Path

project_dir = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker"
)

trips_file = project_dir / "data" / "raw" / "hyderabad" / "trips.txt"

trips = pd.read_csv(trips_file, dtype=str)

target_trips = ["41738901", "41738902"]

columns_to_show = [
    col for col in [
        "route_id",
        "service_id",
        "trip_id",
        "trip_headsign",
        "direction_id",
        "shape_id",
        "block_id"
    ]
    if col in trips.columns
]

display(
    trips.loc[
        trips["trip_id"].isin(target_trips),
        columns_to_show
    ]
)


,route_id,service_id,trip_id,direction_id
10402,300,MTWTFSS,41738901,1
10739,300,MTWTFSS,41738902,0


In [52]:
display(
    trips.loc[
        trips["trip_id"].isin(["41738901", "41738902"])
    ].head()
)

,route_id,trip_id,service_id,direction_id,trip_short_name
10402,300,41738901,MTWTFSS,1,300-UPPAL DEPOT-TO-MEHDIPATNAM-UPL
10739,300,41738902,MTWTFSS,0,300-MEHDIPATNAM-TO-JBS-UPL


In [53]:
import pandas as pd
from pathlib import Path

project_dir = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker"
)

stop_times_file = (
    project_dir / "data" / "raw" / "hyderabad" / "stop_times.txt"
)

target_trips = ["41738901", "41738902"]

st = pd.read_csv(
    stop_times_file,
    usecols=["trip_id", "stop_sequence", "arrival_time", "departure_time"],
    dtype={"trip_id": str}
)

st = st[st["trip_id"].isin(target_trips)].copy()

def gtfs_seconds(value):
    h, m, s = map(int, value.split(":"))
    return h * 3600 + m * 60 + s

st["arrival_seconds"] = st["arrival_time"].map(gtfs_seconds)
st["previous_departure_seconds"] = (
    st.sort_values(["trip_id", "stop_sequence"])
      .groupby("trip_id")["departure_time"]
      .transform(lambda x: x.map(gtfs_seconds).shift())
)

st["gap_minutes"] = (
    st["arrival_seconds"] - st["previous_departure_seconds"]
) / 60

display(
    st.sort_values("gap_minutes", ascending=False)
      .groupby("trip_id")
      .head(10)[
          ["trip_id", "stop_sequence", "arrival_time",
           "departure_time", "gap_minutes"]
      ]
      .sort_values(["trip_id", "gap_minutes"], ascending=[True, False])
)

,trip_id,stop_sequence,arrival_time,departure_time,gap_minutes
287396,41738901,73,243:13:38,243:13:38,430.000000
287395,41738901,72,236:03:38,236:03:38,419.883333
287394,41738901,71,229:03:45,229:03:45,405.983333
287393,41738901,70,222:17:46,222:17:46,393.566667
287392,41738901,69,215:44:12,215:44:12,387.883333
287391,41738901,68,209:16:19,209:16:19,380.350000
287390,41738901,67,202:55:58,202:55:58,376.816667
287389,41738901,66,196:39:09,196:39:09,373.100000
287388,41738901,65,190:26:03,190:26:03,367.216667
287387,41738901,64,184:18:50,184:18:50,363.900000


In [54]:
import pandas as pd
from pathlib import Path

project_dir = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker"
)

stop_times_file = (
    project_dir / "data" / "raw" / "hyderabad" / "stop_times.txt"
)

st = pd.read_csv(stop_times_file, dtype={"trip_id": str})

for trip_id in ["41738901", "41738902"]:
    print(f"\nTrip ID: {trip_id}")

    trip = (
        st[st["trip_id"] == trip_id]
        .sort_values("stop_sequence")
    )

    display(
        trip.iloc[
            list(range(min(3, len(trip)))) +
            list(range(max(3, len(trip) - 3), len(trip)))
        ][
            ["trip_id", "stop_id", "stop_sequence",
             "arrival_time", "departure_time"]
        ]
    )


Trip ID: 41738901


,trip_id,stop_id,stop_sequence,arrival_time,departure_time
287324,41738901,hCkrCbOE,1,06:25:00,06:25:00
287325,41738901,bGRebiXX,2,06:26:39,06:26:39
287326,41738901,G3Zpsr6s,3,06:31:43,06:31:43
287394,41738901,pWgyJcJa,71,229:03:45,229:03:45
287395,41738901,WZDC0taq,72,236:03:38,236:03:38
287396,41738901,U6BONCXR,73,243:13:38,243:13:38



Trip ID: 41738902


,trip_id,stop_id,stop_sequence,arrival_time,departure_time
299122,41738902,NUmq3A4e,1,08:05:00,08:05:00
299123,41738902,5bo1bbrm,2,08:08:24,08:08:24
299124,41738902,LeH2BEeI,3,08:14:34,08:14:34
299206,41738902,54aomdtX,85,297:12:38,297:12:38
299207,41738902,Z30Pa3qu,86,304:32:58,304:32:58
299208,41738902,GTRQRJwL,87,311:57:58,311:57:58


In [55]:
import pandas as pd
from pathlib import Path

project_dir = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker"
)

stop_times_file = (
    project_dir / "data" / "raw" / "hyderabad" / "stop_times.txt"
)

st = pd.read_csv(
    stop_times_file,
    usecols=[
        "trip_id",
        "stop_sequence",
        "arrival_time",
        "departure_time"
    ],
    dtype={"trip_id": str}
)

st = st[st["trip_id"].isin(["41738901", "41738902"])].copy()
st = st.sort_values(["trip_id", "stop_sequence"])

def gtfs_seconds(value):
    h, m, s = map(int, value.split(":"))
    return h * 3600 + m * 60 + s

st["arrival_seconds"] = st["arrival_time"].map(gtfs_seconds)
st["previous_departure_seconds"] = (
    st.groupby("trip_id")["departure_time"]
      .transform(lambda values: values.map(gtfs_seconds).shift())
)

st["gap_minutes"] = (
    st["arrival_seconds"] - st["previous_departure_seconds"]
) / 60

gap_summary = (
    st.groupby("trip_id")
      .agg(
          total_stops=("stop_sequence", "count"),
          gaps_over_60_min=("gap_minutes", lambda x: (x > 60).sum()),
          maximum_gap_minutes=("gap_minutes", "max"),
          negative_gaps=("gap_minutes", lambda x: (x < 0).sum())
      )
      .reset_index()
)

display(gap_summary)

,trip_id,total_stops,gaps_over_60_min,maximum_gap_minutes,negative_gaps
0,41738901,73,58,430.0,0
1,41738902,87,68,445.0,0


In [56]:
import pandas as pd
from pathlib import Path

project_dir = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker"
)

stop_times_file = (
    project_dir / "data" / "raw" / "hyderabad" / "stop_times.txt"
)

st = pd.read_csv(
    stop_times_file,
    usecols=["trip_id", "stop_sequence", "arrival_time", "departure_time"],
    dtype={"trip_id": str}
)

def gtfs_seconds(value):
    h, m, s = map(int, value.split(":"))
    return h * 3600 + m * 60 + s

st = st.sort_values(["trip_id", "stop_sequence"])
st["arrival_seconds"] = st["arrival_time"].map(gtfs_seconds)

st["previous_departure_seconds"] = (
    st.groupby("trip_id")["departure_time"]
      .transform(lambda x: x.map(gtfs_seconds).shift())
)

st["gap_minutes"] = (
    st["arrival_seconds"] - st["previous_departure_seconds"]
) / 60

trip_summary = (
    st.groupby("trip_id")
      .agg(
          total_gaps=("gap_minutes", "count"),
          gaps_over_60_min=("gap_minutes", lambda x: (x > 60).sum()),
          maximum_gap_minutes=("gap_minutes", "max")
      )
)

affected = trip_summary[trip_summary["gaps_over_60_min"] > 0]

print("Total trips checked:", len(trip_summary))
print("Trips with gaps over 60 minutes:", len(affected))
print("Total gaps over 60 minutes:", int(affected["gaps_over_60_min"].sum()))

display(affected.sort_values("maximum_gap_minutes", ascending=False).head(10))

Total trips checked: 44892
Trips with gaps over 60 minutes: 20039
Total gaps over 60 minutes: 196511


,total_gaps,gaps_over_60_min,maximum_gap_minutes
trip_id,,,
41753000,5,4,725.0
41777461,9,9,690.0
41738902,86,68,445.0
41738901,72,58,430.0
41752201,15,13,420.0
923698,17,15,410.0
1217841,54,29,265.0
1214724,29,24,250.0
1209178,32,24,245.0


In [57]:

gap_distribution = pd.Series({
    "Over 60 minutes": (st["gap_minutes"] > 60).sum(),
    "Over 120 minutes": (st["gap_minutes"] > 120).sum(),
    "Over 240 minutes": (st["gap_minutes"] > 240).sum(),
    "Over 360 minutes": (st["gap_minutes"] > 360).sum(),
    "Negative gaps": (st["gap_minutes"] < 0).sum(),
})

display(gap_distribution.to_frame("gap_count"))

,gap_count
Over 60 minutes,196511
Over 120 minutes,6319
Over 240 minutes,102
Over 360 minutes,52
Negative gaps,0


In [58]:
from pathlib import Path
import pandas as pd

project_dir = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker"
)

reports_dir = project_dir / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

gap_thresholds = [60, 120, 240, 360]

summary = pd.DataFrame({
    "threshold_minutes": gap_thresholds,
    "gap_count": [
        int((st["gap_minutes"] > threshold).sum())
        for threshold in gap_thresholds
    ]
})

summary["city"] = "Hyderabad"
summary["interpretation"] = (
    "Flag for source verification; not automatically an error"
)

output_file = reports_dir / "hyderabad_gap_distribution_summary.csv"
summary.to_csv(output_file, index=False)

print("Report saved to:", output_file)
display(summary)

Report saved to: C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\reports\hyderabad_gap_distribution_summary.csv


,threshold_minutes,gap_count,city,interpretation
0,60,196511,Hyderabad,Flag for source verification; not automaticall...
1,120,6319,Hyderabad,Flag for source verification; not automaticall...
2,240,102,Hyderabad,Flag for source verification; not automaticall...
3,360,52,Hyderabad,Flag for source verification; not automaticall...


In [59]:
from pathlib import Path
import pandas as pd

project_dir = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker"
)
processed_dir = project_dir / "data" / "processed"

feature_files = sorted(
    processed_dir.glob("*_trip_headway_features.csv")
)

for file in feature_files:
    df = pd.read_csv(file)

    print(f"\nCity file: {file.name}")
    print("Total records:", len(df))
    print("Time-period categories:")
    display(
        df["time_period"]
        .value_counts(dropna=False)
        .rename_axis("time_period")
        .reset_index(name="record_count")
    )


City file: ahmedabad_trip_headway_features.csv
Total records: 14323
Time-period categories:


,time_period,record_count
0,Morning,5812
1,Afternoon,4207
2,Evening,3323
3,Night,981



City file: chennai_trip_headway_features.csv
Total records: 48435
Time-period categories:


,time_period,record_count
0,Morning,19003
1,Afternoon,14717
2,Evening,10588
3,Night,4127



City file: hyderabad_trip_headway_features.csv
Total records: 44892
Time-period categories:


,time_period,record_count
0,Morning,18435
1,Afternoon,13769
2,Evening,10401
3,Night,2287



City file: mumbai_trip_headway_features.csv
Total records: 99080
Time-period categories:


,time_period,record_count
0,Morning,38043
1,Afternoon,25447
2,Evening,21958
3,Night,13632



City file: pune_trip_headway_features.csv
Total records: 10128
Time-period categories:


,time_period,record_count
0,Morning,6917
1,Afternoon,2870
2,Evening,215
3,Night,126


In [60]:
from pathlib import Path
import pandas as pd

project_dir = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker"
)
processed_dir = project_dir / "data" / "processed"

feature_files = sorted(
    processed_dir.glob("*_trip_headway_features.csv")
)

for file in feature_files:
    df = pd.read_csv(file)

    print(f"\n--- {file.stem} ---")

    display(
        pd.crosstab(
            df["clock_hour"],
            df["time_period"],
            margins=True
        )
    )


--- ahmedabad_trip_headway_features ---


time_period,Afternoon,Evening,Morning,Night,All
clock_hour,,,,,
5,0,0,210,0,210
6,0,0,827,0,827
7,0,0,1072,0,1072
8,0,0,1055,0,1055
9,0,0,925,0,925
10,0,0,888,0,888
11,0,0,835,0,835
12,779,0,0,0,779
13,739,0,0,0,739



--- chennai_trip_headway_features ---


time_period,Afternoon,Evening,Morning,Night,All
clock_hour,,,,,
0,0,0,0,105,105
1,0,0,0,105,105
2,0,0,0,103,103
3,0,0,0,203,203
4,0,0,0,870,870
5,0,0,2168,0,2168
6,0,0,3012,0,3012
7,0,0,2807,0,2807
8,0,0,2855,0,2855



--- hyderabad_trip_headway_features ---


time_period,Afternoon,Evening,Morning,Night,All
clock_hour,,,,,
0,0,0,0,22,22
1,0,0,0,14,14
2,0,0,0,15,15
3,0,0,0,74,74
4,0,0,0,552,552
5,0,0,1634,0,1634
6,0,0,2937,0,2937
7,0,0,3183,0,3183
8,0,0,2927,0,2927



--- mumbai_trip_headway_features ---


time_period,Afternoon,Evening,Morning,Night,All
clock_hour,,,,,
0,0,0,0,1133,1133
1,0,0,0,507,507
2,0,0,0,121,121
3,0,0,0,110,110
4,0,0,0,609,609
5,0,0,2166,0,2166
6,0,0,5208,0,5208
7,0,0,6384,0,6384
8,0,0,6485,0,6485



--- pune_trip_headway_features ---


time_period,Afternoon,Evening,Morning,Night,All
clock_hour,,,,,
0,0,0,0,7,7
1,0,0,0,9,9
2,0,0,0,7,7
3,0,0,0,8,8
4,0,0,0,46,46
5,0,0,418,0,418
6,0,0,921,0,921
7,0,0,1118,0,1118
8,0,0,1186,0,1186


In [61]:
from pathlib import Path
import pandas as pd

project_dir = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker"
)
processed_dir = project_dir / "data" / "processed"

feature_files = sorted(
    processed_dir.glob("*_trip_headway_features.csv")
)

def parse_gtfs_time(value):
    h, m, s = map(int, value.split(":"))
    return h * 3600 + m * 60 + s

def expected_period(hour):
    if 5 <= hour <= 11:
        return "Morning"
    elif 12 <= hour <= 16:
        return "Afternoon"
    elif 17 <= hour <= 20:
        return "Evening"
    else:
        return "Night"

for file in feature_files:
    df = pd.read_csv(file)

    seconds = df["departure_time"].map(parse_gtfs_time)
    expected_clock_hour = (seconds // 3600) % 24
    expected_minute = (seconds // 60) % 60
    expected_time_period = expected_clock_hour.map(expected_period)

    print(f"\n{file.name}")
    print("Rows:", len(df))
    print("Clock-hour mismatches:",
          int((df["clock_hour"] != expected_clock_hour).sum()))
    print("Minute mismatches:",
          int((df["minute"] != expected_minute).sum()))
    print("Time-period mismatches:",
          int((df["time_period"] != expected_time_period).sum()))


ahmedabad_trip_headway_features.csv
Rows: 14323
Clock-hour mismatches: 0
Minute mismatches: 0
Time-period mismatches: 0

chennai_trip_headway_features.csv
Rows: 48435
Clock-hour mismatches: 0
Minute mismatches: 0
Time-period mismatches: 0

hyderabad_trip_headway_features.csv
Rows: 44892
Clock-hour mismatches: 0
Minute mismatches: 0
Time-period mismatches: 0

mumbai_trip_headway_features.csv
Rows: 99080
Clock-hour mismatches: 0
Minute mismatches: 0
Time-period mismatches: 0

pune_trip_headway_features.csv
Rows: 10128
Clock-hour mismatches: 0
Minute mismatches: 0
Time-period mismatches: 0


In [62]:
from pathlib import Path
import pandas as pd

project_dir = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker"
)
processed_dir = project_dir / "data" / "processed"

feature_files = sorted(
    processed_dir.glob("*_trip_headway_features.csv")
)

for file in feature_files:
    df = pd.read_csv(file)

    df = df.sort_values(
        ["route_id", "service_id", "departure_time_seconds"]
    ).copy()

    df["expected_headway"] = (
        df.groupby(["route_id", "service_id"])[
            "departure_time_seconds"
        ].diff() / 60
    )

    missing_headway = df["headway_minutes"].isna()
    expected_missing = df["expected_headway"].isna()

    print(f"\n{file.name}")
    print("Total records:", len(df))
    print("Missing headways:", int(missing_headway.sum()))
    print("Missing expected headways:", int(expected_missing.sum()))
    print(
        "Missingness mismatches:",
        int((missing_headway != expected_missing).sum())
    )
    print(
        "Headway value mismatches:",
        int(
            (
                df["headway_minutes"].notna()
                & df["expected_headway"].notna()
                & (
                    (df["headway_minutes"] - df["expected_headway"]).abs()
                    > 1e-9
                )
            ).sum()
        )
    )


ahmedabad_trip_headway_features.csv
Total records: 14323
Missing headways: 633
Missing expected headways: 633
Missingness mismatches: 0
Headway value mismatches: 0

chennai_trip_headway_features.csv
Total records: 48435
Missing headways: 4732
Missing expected headways: 4732
Missingness mismatches: 0
Headway value mismatches: 0

hyderabad_trip_headway_features.csv
Total records: 44892
Missing headways: 1031
Missing expected headways: 1031
Missingness mismatches: 0
Headway value mismatches: 0

mumbai_trip_headway_features.csv
Total records: 99080
Missing headways: 612
Missing expected headways: 612
Missingness mismatches: 0
Headway value mismatches: 0

pune_trip_headway_features.csv
Total records: 10128
Missing headways: 315
Missing expected headways: 315
Missingness mismatches: 0
Headway value mismatches: 0


In [63]:
from pathlib import Path
import pandas as pd

project_dir = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker"
)
processed_dir = project_dir / "data" / "processed"

feature_files = sorted(
    processed_dir.glob("*_trip_headway_features.csv")
)

for file in feature_files:
    df = pd.read_csv(file)

    zero_headways = (
        df["headway_minutes"].notna()
        & (df["headway_minutes"] == 0)
    )

    duplicate_departures = df.duplicated(
        subset=[
            "route_id",
            "service_id",
            "departure_time_seconds"
        ],
        keep=False
    )

    print(f"\n{file.name}")
    print("Zero-minute headways:", int(zero_headways.sum()))
    print(
        "Records sharing a departure time within route/service:",
        int(duplicate_departures.sum())
    )


ahmedabad_trip_headway_features.csv
Zero-minute headways: 933
Records sharing a departure time within route/service: 1866

chennai_trip_headway_features.csv
Zero-minute headways: 1005
Records sharing a departure time within route/service: 1984

hyderabad_trip_headway_features.csv
Zero-minute headways: 13462
Records sharing a departure time within route/service: 21000

mumbai_trip_headway_features.csv
Zero-minute headways: 9257
Records sharing a departure time within route/service: 18491

pune_trip_headway_features.csv
Zero-minute headways: 1507
Records sharing a departure time within route/service: 3014


In [64]:
from pathlib import Path
import pandas as pd

project_dir = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker"
)
processed_dir = project_dir / "data" / "processed"

feature_files = sorted(
    processed_dir.glob("*_trip_headway_features.csv")
)

for file in feature_files:
    df = pd.read_csv(file)

    group_cols = [
        "route_id",
        "service_id",
        "departure_time_seconds"
    ]

    group_sizes = (
        df.groupby(group_cols, dropna=False)
        .size()
        .reset_index(name="trip_count")
    )

    duplicates = group_sizes[
        group_sizes["trip_count"] > 1
    ].sort_values("trip_count", ascending=False)

    print(f"\n{'=' * 65}")
    print(file.name)
    print("Distinct duplicate departure groups:", len(duplicates))
    print(
        "Records in duplicate groups:",
        int(duplicates["trip_count"].sum())
    )
    print("Largest duplicate groups:")
    print(duplicates.head(5).to_string(index=False))


ahmedabad_trip_headway_features.csv
Distinct duplicate departure groups: 933
Records in duplicate groups: 1866
Largest duplicate groups:
route_id    service_id  departure_time_seconds  trip_count
   103-2 1,2,3,4,5,6,7                   76800           2
     105 1,2,3,4,5,6,7                   21900           2
     105 1,2,3,4,5,6,7                   23700           2
     105 1,2,3,4,5,6,7                   24300           2
     105 1,2,3,4,5,6,7                   24900           2

chennai_trip_headway_features.csv
Distinct duplicate departure groups: 979
Records in duplicate groups: 1984
Largest duplicate groups:
 route_id  service_id  departure_time_seconds  trip_count
MTC-22606 MTC-Regular                   51900           3
MTC-18782 MTC-Regular                   39300           3
MTC-20580 MTC-Regular                   51300           3
MTC-20089 MTC-Regular                   43800           3
MTC-22850 MTC-Regular                   25200           3

hyderabad_trip_headway_

In [65]:
from pathlib import Path
import pandas as pd

project_dir = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker"
)
processed_dir = project_dir / "data" / "processed"

feature_files = sorted(
    processed_dir.glob("*_trip_headway_features.csv")
)

for file in feature_files:
    df = pd.read_csv(file)

    duplicate_time = df.duplicated(
        subset=[
            "route_id",
            "service_id",
            "departure_time_seconds"
        ],
        keep=False
    )

    duplicate_trip_ids = df["trip_id"].duplicated(keep=False)

    print(f"\n{'=' * 60}")
    print(file.name)
    print("Total records:", len(df))
    print("Records with repeated trip_id:", int(duplicate_trip_ids.sum()))
    print(
        "Repeated-departure records with repeated trip_id:",
        int((duplicate_time & duplicate_trip_ids).sum())
    )

    print("\nSample repeated departures:")
    print(
        df.loc[
            duplicate_time,
            [
                "route_id",
                "service_id",
                "departure_time",
                "departure_time_seconds",
                "trip_id",
                "headway_minutes"
            ]
        ]
        .sort_values(
            ["route_id", "service_id", "departure_time_seconds"]
        )
        .head(10)
        .to_string(index=False)
    )


ahmedabad_trip_headway_features.csv
Total records: 14323
Records with repeated trip_id: 0
Repeated-departure records with repeated trip_id: 0

Sample repeated departures:
route_id    service_id departure_time  departure_time_seconds     trip_id  headway_minutes
   103-2 1,2,3,4,5,6,7       21:20:00                   76800 AMC_9348974             55.0
   103-2 1,2,3,4,5,6,7       21:20:00                   76800 AMC_9348986              0.0
     105 1,2,3,4,5,6,7       06:05:00                   21900 AMC_9348992             10.0
     105 1,2,3,4,5,6,7       06:05:00                   21900 AMC_9349187              0.0
     105 1,2,3,4,5,6,7       06:35:00                   23700 AMC_9349146             10.0
     105 1,2,3,4,5,6,7       06:35:00                   23700 AMC_9349229              0.0
     105 1,2,3,4,5,6,7       06:45:00                   24300 AMC_9349005             10.0
     105 1,2,3,4,5,6,7       06:45:00                   24300 AMC_9349160              0.0
     105 

In [66]:
from pathlib import Path
import pandas as pd

project_dir = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker"
)
processed_dir = project_dir / "data" / "processed"
raw_dir = project_dir / "data" / "raw"

feature_files = sorted(
    processed_dir.glob("*_trip_headway_features.csv")
)

for file in feature_files:
    city = file.name.replace(
        "_trip_headway_features.csv", ""
    )
    trips_file = raw_dir / city / "trips.txt"

    print(f"\n{'=' * 60}")
    print("City:", city)

    if not trips_file.exists():
        print("Raw trips.txt not found:", trips_file)
        continue

    # Read trip IDs as text in both files
    features = pd.read_csv(
        file, dtype={"trip_id": str}
    )
    trips = pd.read_csv(
        trips_file,
        dtype=str,
        usecols=lambda col: col in [
            "trip_id", "direction_id"
        ]
    )

    if "direction_id" not in trips.columns:
        print("direction_id is not available")
        continue

    direction_map = trips[
        ["trip_id", "direction_id"]
    ].drop_duplicates(subset=["trip_id"])

    checked = features.merge(
        direction_map,
        on="trip_id",
        how="left",
        validate="many_to_one"
    )

    group_cols = [
        "route_id",
        "service_id",
        "departure_time_seconds"
    ]

    repeated_before = checked.duplicated(
        subset=group_cols, keep=False
    )

    matched = checked[
        checked["direction_id"].notna()
    ].copy()

    repeated_after = matched.duplicated(
        subset=group_cols + ["direction_id"],
        keep=False
    )

    print("Total records:", len(checked))
    print(
        "Repeated-departure records before direction check:",
        int(repeated_before.sum())
    )
    print(
        "Repeated-departure records after direction check "
        "(matched directions only):",
        int(repeated_after.sum())
    )
    print(
        "Records without matched direction_id:",
        int(checked["direction_id"].isna().sum())
    )


City: ahmedabad
Total records: 14323
Repeated-departure records before direction check: 1866
Repeated-departure records after direction check (matched directions only): 16
Records without matched direction_id: 0

City: chennai
Total records: 48435
Repeated-departure records before direction check: 1984
Repeated-departure records after direction check (matched directions only): 1984
Records without matched direction_id: 1388

City: hyderabad
Total records: 44892
Repeated-departure records before direction check: 21000
Repeated-departure records after direction check (matched directions only): 15923
Records without matched direction_id: 0

City: mumbai
Total records: 99080
Repeated-departure records before direction check: 18491
Repeated-departure records after direction check (matched directions only): 194
Records without matched direction_id: 0

City: pune
Total records: 10128
Repeated-departure records before direction check: 3014
Repeated-departure records after direction check (mat

In [67]:
from pathlib import Path
import pandas as pd

project_dir = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker"
)

features_file = (
    project_dir / "data" / "processed"
    / "chennai_trip_headway_features.csv"
)
trips_file = (
    project_dir / "data" / "raw"
    / "chennai" / "trips.txt"
)

features = pd.read_csv(
    features_file, dtype=str
)
trips = pd.read_csv(
    trips_file, dtype=str, usecols=["trip_id"]
)

# Normalize whitespace for comparison only
feature_ids = features["trip_id"].str.strip()
raw_ids = trips["trip_id"].str.strip()

unmatched_mask = ~feature_ids.isin(set(raw_ids))
unmatched_ids = feature_ids[unmatched_mask]

print("Feature records:", len(features))
print("Raw GTFS trip records:", len(trips))
print("Unique feature trip IDs:", feature_ids.nunique())
print("Unique raw trip IDs:", raw_ids.nunique())
print("Feature records not matched to raw trips:", int(unmatched_mask.sum()))
print("Unique unmatched trip IDs:", unmatched_ids.nunique())

print("\nExamples of unmatched feature trip IDs:")
print(unmatched_ids.drop_duplicates().head(20).to_list())

print("\nExamples of raw GTFS trip IDs:")
print(raw_ids.drop_duplicates().head(20).to_list())

Feature records: 48435
Raw GTFS trip records: 48435
Unique feature trip IDs: 48435
Unique raw trip IDs: 48435
Feature records not matched to raw trips: 0
Unique unmatched trip IDs: 0

Examples of unmatched feature trip IDs:
[]

Examples of raw GTFS trip IDs:
['MTC-71695', 'MTC-120662', 'MTC-119010', 'MTC-119011', 'MTC-120222', 'MTC-120663', 'MTC-120019', 'MTC-119009', 'MTC-23080', 'MTC-72413', 'MTC-120664', 'MTC-119012', 'MTC-87040', 'MTC-119013', 'MTC-22225', 'MTC-120112', 'MTC-103482', 'MTC-120202', 'MTC-32201', 'MTC-120073']


In [68]:

from pathlib import Path
import pandas as pd

# Project folders
project_dir = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker"
)
processed_dir = project_dir / "data" / "processed"
reports_dir = project_dir / "reports"

reports_dir.mkdir(parents=True, exist_ok=True)

feature_files = sorted(
    processed_dir.glob("*_trip_headway_features.csv")
)

if not feature_files:
    raise FileNotFoundError(
        f"No feature CSV files found in {processed_dir}"
    )

summary = []

required_columns = [
    "city",
    "trip_id",
    "route_id",
    "service_id",
    "departure_time",
    "departure_time_seconds",
    "service_hour",
    "clock_hour",
    "minute",
    "time_period",
    "headway_minutes",
]

for file in feature_files:
    df = pd.read_csv(file)

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        print(f"\n{file.name}: missing columns {missing_columns}")
        continue

    summary.append({
        "city": file.name.replace(
            "_trip_headway_features.csv", ""
        ),
        "total_records": len(df),
        "unique_trip_ids": df["trip_id"].nunique(),
        "duplicate_trip_ids": int(
            df["trip_id"].duplicated().sum()
        ),
        "missing_values_total": int(
            df.isna().sum().sum()
        ),
        "missing_headways": int(
            df["headway_minutes"].isna().sum()
        ),
        "zero_headways": int(
            (df["headway_minutes"] == 0).sum()
        ),
        "negative_headways": int(
            (df["headway_minutes"] < 0).sum()
        ),
        "invalid_clock_hours": int(
            (~df["clock_hour"].between(0, 23)).sum()
        ),
        "invalid_minutes": int(
            (~df["minute"].between(0, 59)).sum()
        ),
        "missing_required_columns": 0,
    })

summary_df = pd.DataFrame(summary)

report_path = (
    reports_dir / "module2_final_validation_summary.csv"
)

summary_df.to_csv(report_path, index=False)

print("\nMODULE 2 FINAL VALIDATION")
print("=" * 70)
print(summary_df.to_string(index=False))
print("\nReport saved to:")
print(report_path)



MODULE 2 FINAL VALIDATION
     city  total_records  unique_trip_ids  duplicate_trip_ids  missing_values_total  missing_headways  zero_headways  negative_headways  invalid_clock_hours  invalid_minutes  missing_required_columns
ahmedabad          14323            14323                   0                   633               633            933                  0                    0                0                         0
  chennai          48435            48435                   0                  4732              4732           1005                  0                    0                0                         0
hyderabad          44892            44892                   0                  1031              1031          13462                  0                    0                0                         0
   mumbai          99080            99080                   0                   612               612           9257                  0                    0                0

In [69]:
from pathlib import Path

project = Path.cwd()
reports = project / "reports"
processed = project / "data" / "processed"

files_to_check = [
    reports / "module2_feature_dictionary.csv",
    reports / "module2_data_quality_report.csv",
    reports / "module2_final_validation_summary.csv",
    reports / "module2_assumptions_and_limitations.md",
    processed / "route_features.csv",
    processed / "route_profiles.csv",
]

for file in files_to_check:
    print(("FOUND  " if file.exists() else "MISSING") + str(file))

FOUND  C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\reports\module2_feature_dictionary.csv
FOUND  C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\reports\module2_data_quality_report.csv
FOUND  C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\reports\module2_final_validation_summary.csv
FOUND  C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\reports\module2_assumptions_and_limitations.md
FOUND  C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\data\processed\route_features.csv
FOUND  C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\data\processed\route_profiles.csv


import pandas as pd
from pathlib import Path

file = Path.cwd() / "reports" / "module2_final_validation_summary.csv"
validation = pd.read_csv(file)

print("Validation report shape:", validation.shape)
display(validation)

### Validation Results and Interpretation

The headway validation was performed for five cities: Ahmedabad, Chennai, Hyderabad, Mumbai, and Pune. A total of 216,858 records were checked. The results showed no duplicate trip IDs, negative headways, invalid clock hours, invalid minute values, or missing required columns.

A total of 7,323 records contain missing headway values. These records require further interpretation because a headway may be unavailable for the first departure or when the previous departure is not recorded. In addition, 26,164 records contain zero headways, which may indicate simultaneous scheduled departures. These values should be checked against the source GTFS data before any correction or removal.

Overall, the validation confirms that the checked records satisfy the listed structural checks. However, further investigation of missing and zero headways is necessary before using them for route-frequency analysis. The original source data should be preserved, and any processing decisions should be documented.


In [70]:

from pathlib import Path
import pandas as pd

project = Path.cwd()
features_file = project / "data" / "processed" / "route_features.csv"

print("Project folder:", project)
print("Feature file exists:", features_file.exists())

if features_file.exists():
    features = pd.read_csv(features_file)

    print("\nRoute features shape:", features.shape)
    print("\nColumns:")
    print(features.columns.tolist())

    print("\nFirst five rows:")
    display(features.head())

    print("\nRoute count by city:")
    display(features.groupby("city").size().rename("route_count").to_frame())
else:
    print("Please check that Jupyter is running in the RouteRanker project folder.")


Project folder: C:\Users\LENOVO\OneDrive\Desktop\RouteRanker
Feature file exists: True

Route features shape: (8322, 20)

Columns:
['route_id', 'route_short_name', 'route_long_name', 'route_type', 'scheduled_trip_count', 'stop_count', 'estimated_straight_line_route_km', 'trips_with_distance', 'operates_monday', 'operates_tuesday', 'operates_wednesday', 'operates_thursday', 'operates_friday', 'operates_saturday', 'operates_sunday', 'operating_day_count', 'city', 'start_service_hour', 'end_service_hour', 'trips_with_valid_departure']

First five rows:


,route_id,route_short_name,route_long_name,route_type,scheduled_trip_count,stop_count,estimated_straight_line_route_km,trips_with_distance,operates_monday,operates_tuesday,operates_wednesday,operates_thursday,operates_friday,operates_saturday,operates_sunday,operating_day_count,city,start_service_hour,end_service_hour,trips_with_valid_departure
0,101sh,101sh,P.D.P.U. Cross Road ⇆ Asarva Railway Station,3,10,44,44.528570,10.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,7.0,ahmedabad,6.0,19.0,10
1,103-1,103,Bhakti Circle ⇆ Gujarat University,3,22,80,22.365598,22.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,7.0,ahmedabad,6.0,21.0,22
2,103-2,103,Bhakti Circle ⇆ Sarangpur,3,14,52,14.922204,14.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,7.0,ahmedabad,5.0,23.0,14
3,103-3,103,Gujarat University ⇆ Sarangpur,3,6,28,8.947943,6.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,7.0,ahmedabad,12.0,22.0,6
4,105,105,Naroda Industrial Township( MuthiyaGam ) ⇆ Lal...,3,216,40,63.726733,216.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,7.0,ahmedabad,5.0,23.0,216



Route count by city:


,route_count
city,
ahmedabad,639
chennai,5402
hyderabad,1031
mumbai,935
pune,315


In [71]:

import pandas as pd
from pathlib import Path

file = Path.cwd() / "data" / "processed" / "route_features.csv"
df = pd.read_csv(file)

print("Total route records:", len(df))
print("Routes with scheduled trips:",
      int((df["scheduled_trip_count"] > 0).sum()))
print("Routes without scheduled trips:",
      int((df["scheduled_trip_count"] == 0).sum()))
print("Missing scheduled trip counts:",
      int(df["scheduled_trip_count"].isna().sum()))

df["route_status"] = df["scheduled_trip_count"].apply(
    lambda x: "Missing trip count" if pd.isna(x)
    else "Active" if x > 0
    else "No scheduled trips"
)

print("\nRoute status by city:")
display(pd.crosstab(df["city"], df["route_status"]))


Total route records: 8322
Routes with scheduled trips: 7316
Routes without scheduled trips: 1006
Missing scheduled trip counts: 0

Route status by city:


route_status,Active,No scheduled trips
city,,
ahmedabad,633,6
chennai,4725,677
hyderabad,1031,0
mumbai,612,323
pune,315,0



### Route Status Analysis

The route feature dataset contains 8,322 route records across five cities. Among these, 7,316 routes have scheduled trips, while 1,006 routes have no scheduled trips in the processed data.

The city-wise analysis identifies 677 routes without scheduled trips in Chennai, 323 in Mumbai, and 6 in Ahmedabad. Hyderabad and Pune have no routes in this category.

Routes without scheduled trips are retained in the dataset for review. Their absence from the trip data does not automatically mean that they are inefficient or unused. They should be investigated before being included in route-efficiency analysis.


In [72]:
from pathlib import Path
import pandas as pd

folder = Path.cwd() / "data" / "processed"

for file in sorted(folder.glob("*_trip_time_features.csv")):
    df = pd.read_csv(file)

    print("\nCITY:", file.name)
    print("Rows:", len(df))

    for col in ["departure_time", "clock_hour", "minute", "time_period"]:
        print(f"{col}: {'FOUND' if col in df.columns else 'MISSING'}")

    if all(c in df.columns for c in ["departure_time", "clock_hour", "minute", "time_period"]):
        print("Sample:")
        display(df[["departure_time", "clock_hour", "minute", "time_period"]].head(3))


CITY: ahmedabad_trip_time_features.csv
Rows: 14323
departure_time: FOUND
clock_hour: FOUND
minute: FOUND
time_period: FOUND
Sample:


,departure_time,clock_hour,minute,time_period
0,05:15:00,5,15,Morning
1,06:20:00,6,20,Morning
2,08:20:00,8,20,Morning



CITY: chennai_trip_time_features.csv
Rows: 48435
departure_time: FOUND
clock_hour: FOUND
minute: FOUND
time_period: FOUND
Sample:


,departure_time,clock_hour,minute,time_period
0,05:27:59,5,27,Morning
1,05:28:23,5,28,Morning
2,05:31:29,5,31,Morning



CITY: hyderabad_trip_time_features.csv
Rows: 44892
departure_time: FOUND
clock_hour: FOUND
minute: FOUND
time_period: FOUND
Sample:


,departure_time,clock_hour,minute,time_period
0,07:20:00,7,20,Morning
1,09:35:00,9,35,Morning
2,11:20:00,11,20,Morning



CITY: mumbai_trip_time_features.csv
Rows: 99080
departure_time: FOUND
clock_hour: FOUND
minute: FOUND
time_period: FOUND
Sample:


,departure_time,clock_hour,minute,time_period
0,00:00:00,0,0,Night
1,00:07:00,0,7,Night
2,00:30:00,0,30,Night



CITY: pune_trip_time_features.csv
Rows: 10128
departure_time: FOUND
clock_hour: FOUND
minute: FOUND
time_period: FOUND
Sample:


,departure_time,clock_hour,minute,time_period
0,05:40:00,5,40,Morning
1,06:00:00,6,0,Morning
2,06:40:00,6,40,Morning


In [73]:
from pathlib import Path
import pandas as pd

folder = Path.cwd() / "data" / "processed"

for file in sorted(folder.glob("*_trip_time_features.csv")):
    df = pd.read_csv(file)

    h, m, s = df["departure_time"].str.split(":", expand=True).astype(int).T.values
    expected_hour = h % 24
    expected_minute = m

    print(
        file.name,
        "| Rows:", len(df),
        "| Hour mismatches:", int((df["clock_hour"] != expected_hour).sum()),
        "| Minute mismatches:", int((df["minute"] != expected_minute).sum())
    )

ahmedabad_trip_time_features.csv | Rows: 14323 | Hour mismatches: 0 | Minute mismatches: 0
chennai_trip_time_features.csv | Rows: 48435 | Hour mismatches: 0 | Minute mismatches: 0
hyderabad_trip_time_features.csv | Rows: 44892 | Hour mismatches: 0 | Minute mismatches: 0
mumbai_trip_time_features.csv | Rows: 99080 | Hour mismatches: 0 | Minute mismatches: 0
pune_trip_time_features.csv | Rows: 10128 | Hour mismatches: 0 | Minute mismatches: 0


In [74]:

from pathlib import Path
import pandas as pd

folder = Path.cwd() / "data" / "processed"

def expected_period(hour):
    if 5 <= hour <= 11:
        return "Morning"
    elif 12 <= hour <= 16:
        return "Afternoon"
    elif 17 <= hour <= 20:
        return "Evening"
    else:
        return "Night"

total_rows = 0
total_mismatches = 0

for file in sorted(folder.glob("*_trip_time_features.csv")):
    df = pd.read_csv(file)
    hours = pd.to_numeric(df["clock_hour"], errors="coerce")
    expected = hours.map(expected_period)
    mismatches = int((df["time_period"] != expected).sum())

    total_rows += len(df)
    total_mismatches += mismatches

    print(f"{file.name}: {mismatches} time-period mismatches")

print("\nTotal records checked:", total_rows)
print("Total mismatches:", total_mismatches)



ahmedabad_trip_time_features.csv: 0 time-period mismatches
chennai_trip_time_features.csv: 0 time-period mismatches
hyderabad_trip_time_features.csv: 0 time-period mismatches
mumbai_trip_time_features.csv: 0 time-period mismatches
pune_trip_time_features.csv: 0 time-period mismatches

Total records checked: 216858
Total mismatches: 0


In [75]:

from pathlib import Path
import pandas as pd

folder = Path.cwd() / "data" / "processed"
features = pd.read_csv(folder / "route_features.csv")
profiles = pd.read_csv(folder / "route_profiles.csv")

print("Route feature records:", len(features))
print("Route profile records:", len(profiles))

print("\nDuplicate route IDs within each city:")
print("Features:", features.duplicated(["city", "route_id"]).sum())
print("Profiles:", profiles.duplicated(["city", "route_id"]).sum())

print("\nMissing route IDs:")
print("Features:", features["route_id"].isna().sum())
print("Profiles:", profiles["route_id"].isna().sum())

print("\nScheduled trip count summary:")
display(features["scheduled_trip_count"].describe())



Route feature records: 8322
Route profile records: 8322

Duplicate route IDs within each city:
Features: 0
Profiles: 0

Missing route IDs:
Features: 0
Profiles: 0

Scheduled trip count summary:


count    8322.000000
mean       26.058399
std        75.377376
min         0.000000
25%         1.000000
50%         4.000000
75%        15.000000
max      2087.000000
Name: scheduled_trip_count, dtype: float64

In [76]:


from pathlib import Path
import pandas as pd

folder = Path.cwd() / "data" / "processed"
df = pd.read_csv(folder / "route_features.csv")

day_columns = [
    "operates_monday",
    "operates_tuesday",
    "operates_wednesday",
    "operates_thursday",
    "operates_friday",
    "operates_saturday",
    "operates_sunday",
]

print("Missing values in operating-day columns:")
print(df[day_columns].isna().sum())

print("\nMissing operating_day_count values:",
      df["operating_day_count"].isna().sum())

print("\nOperating-day values:")
for column in day_columns:
    print(column, sorted(df[column].dropna().unique().tolist()))


Missing values in operating-day columns:
operates_monday       1006
operates_tuesday      1006
operates_wednesday    1006
operates_thursday     1006
operates_friday       1006
operates_saturday     1006
operates_sunday       1006
dtype: int64

Missing operating_day_count values: 1006

Operating-day values:
operates_monday [0.0, 1.0]
operates_tuesday [0.0, 1.0]
operates_wednesday [0.0, 1.0]
operates_thursday [0.0, 1.0]
operates_friday [0.0, 1.0]
operates_saturday [0.0, 1.0]
operates_sunday [0.0, 1.0]


In [77]:

from pathlib import Path
import pandas as pd

folder = Path.cwd() / "data" / "processed"
df = pd.read_csv(folder / "route_features.csv")

missing_days = df["operating_day_count"].isna()
no_trips = df["scheduled_trip_count"].eq(0)

print("Routes with missing operating-day count:", int(missing_days.sum()))
print("Routes with zero scheduled trips:", int(no_trips.sum()))
print("Missing operating days AND zero trips:",
      int((missing_days & no_trips).sum()))
print("Missing operating days BUT trips exist:",
      int((missing_days & ~no_trips).sum()))
print("Operating-day count missing but weekday flags present:",
      int((missing_days & df[[
          "operates_monday", "operates_tuesday",
          "operates_wednesday", "operates_thursday",
          "operates_friday", "operates_saturday",
          "operates_sunday"
      ]].notna().any(axis=1)).sum()))


Routes with missing operating-day count: 1006
Routes with zero scheduled trips: 1006
Missing operating days AND zero trips: 1006
Missing operating days BUT trips exist: 0
Operating-day count missing but weekday flags present: 0


In [78]:

from pathlib import Path
import json

notebook_path = Path.cwd() / "RouteRanker_Module2_Feature_Engineering.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

print("Notebook:", notebook_path.name)
print("Total cells:", len(notebook["cells"]))
print("Code cells:", sum(c["cell_type"] == "code" for c in notebook["cells"]))
print("Markdown cells:", sum(c["cell_type"] == "markdown" for c in notebook["cells"]))
print("Code cells with errors:",
      sum(
          output.get("output_type") == "error"
          for cell in notebook["cells"]
          if cell["cell_type"] == "code"
          for output in cell.get("outputs", [])
      ))


Notebook: RouteRanker_Module2_Feature_Engineering.ipynb
Total cells: 95
Code cells: 85
Markdown cells: 10
Code cells with errors: 0


In [79]:

from pathlib import Path
import shutil

notebook_path = Path.cwd() / "RouteRanker_Module2_Feature_Engineering.ipynb"
backup_path = notebook_path.with_name(
    "RouteRanker_Module2_Feature_Engineering_backup.ipynb"
)

shutil.copy2(notebook_path, backup_path)

print("Backup created:", backup_path.name)
print("Backup exists:", backup_path.exists())


Backup created: RouteRanker_Module2_Feature_Engineering_backup.ipynb
Backup exists: True


In [80]:

from pathlib import Path
import json

notebook_path = Path.cwd() / "RouteRanker_Module2_Feature_Engineering.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

for i, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] == "code":
        source = "".join(cell.get("source", [])).strip()
        first_line = source.splitlines()[0] if source else "[Empty code cell]"
        print(f"Cell {i} (execution {cell.get('execution_count')}): {first_line[:120]}")


Cell 1 (execution 1): from pathlib import Path
Cell 2 (execution 2): cities = ["ahmedabad", "chennai", "hyderabad", "mumbai", "pune"]
Cell 3 (execution 3): for city in cities:
Cell 4 (execution 4): from pathlib import Path
Cell 5 (execution 5): for city in cities:
Cell 6 (execution 6): feature_dict_path = REPORTS_DIR / "module2_feature_dictionary.csv"
Cell 7 (execution 7): quality_path = REPORTS_DIR / "module2_data_quality_report.csv"
Cell 8 (execution 8): assumptions_path = REPORTS_DIR / "module2_assumptions_and_limitations.md"
Cell 9 (execution 9): print("RouteRanker — Module 2: Feature Engineering")
Cell 11 (execution 10): from pathlib import Path
Cell 12 (execution 11): from IPython.display import Code, display
Cell 14 (execution 12): import pandas as pd
Cell 15 (execution 13): print("Headway files found:")
Cell 17 (execution 14): import pandas as pd
Cell 18 (execution 15): import pandas as pd
Cell 19 (execution 16): import pandas as pd
Cell 20 (execution 17): import pandas as pd
C

In [81]:

from pathlib import Path
import json

notebook_path = Path.cwd() / "RouteRanker_Module2_Feature_Engineering.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

for i, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] == "markdown":
        text = "".join(cell.get("source", [])).strip()
        if text:
            print(f"\nCell {i}:\n{text[:350]}")



Cell 0:
# RouteRanker – Module 2: Feature Engineering

## 1. Introduction

RouteRanker is an AI-powered public transit route optimization project that uses General Transit Feed Specification (GTFS) data to understand scheduled transportation services.

Module 2 focuses on transforming raw GTFS data into meaningful features for route analysis, comparison, a

Cell 10:
## 2. Feature Engineering Implementation

The feature-engineering process is implemented in `notebook/module2_feature_engineering.py`.

The script processes GTFS data for Ahmedabad, Chennai, Hyderabad, Mumbai, and Pune. It extracts scheduled trip departure information, derives time-based features, calculates scheduled headways, creates route-level 

Cell 13:
## 3. Module 2 – Results and Observations

### 3.1 Feature Engineering Results

Feature engineering was performed on GTFS public transit data from five Indian cities: Ahmedabad, Chennai, Hyderabad, Mumbai, and Pune. The process generated trip-time features, scheduled h

In [82]:

from pathlib import Path
import json
from collections import defaultdict

notebook_path = Path.cwd() / "RouteRanker_Module2_Feature_Engineering.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

groups = defaultdict(list)

for i, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] == "code":
        source = "".join(cell.get("source", [])).strip()
        if source:
            groups[source].append(i)

print("Exact duplicate code cells:")
found = False

for source, indices in groups.items():
    if len(indices) > 1:
        print(f"Cells {indices}: {source.splitlines()[0][:100]}")
        found = True

if not found:
    print("No exact duplicate code cells found.")


Exact duplicate code cells:
No exact duplicate code cells found.


In [83]:

from pathlib import Path
import json

notebook_path = Path.cwd() / "RouteRanker_Module2_Feature_Engineering.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

empty_code = []
empty_markdown = []

for i, cell in enumerate(notebook["cells"]):
    source = "".join(cell.get("source", [])).strip()

    if not source:
        if cell["cell_type"] == "code":
            empty_code.append(i)
        elif cell["cell_type"] == "markdown":
            empty_markdown.append(i)

print("Empty code cells:", empty_code)
print("Empty Markdown cells:", empty_markdown)



Empty code cells: []
Empty Markdown cells: []


In [84]:
from pathlib import Path
import json

path = Path.cwd() / "RouteRanker_Module2_Feature_Engineering.ipynb"

with open(path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

print("Notebook saved and readable:", path.exists())
print("Total cells:", len(notebook["cells"]))
print("Empty code cells:", sum(
    cell["cell_type"] == "code"
    and not "".join(cell.get("source", [])).strip()
    for cell in notebook["cells"]
))

Notebook saved and readable: True
Total cells: 95
Empty code cells: 0


In [85]:

from pathlib import Path
import pandas as pd

project_dir = Path.cwd()

required_files = [
    "RouteRanker_Module2_Feature_Engineering.ipynb",
    "data/processed/route_features.csv",
    "data/processed/route_profiles.csv",
    "reports/module2_feature_dictionary.csv",
    "reports/module2_data_quality_report.csv",
    "reports/module2_final_validation_summary.csv",
    "reports/module2_assumptions_and_limitations.md",
]

print("FINAL DELIVERABLE CHECK\n")

for item in required_files:
    path = project_dir / item
    print(f"{'PASS' if path.is_file() else 'MISSING'}: {item}")

print("\nDATASET SHAPES")

for filename in [
    "data/processed/route_features.csv",
    "data/processed/route_profiles.csv",
]:
    path = project_dir / filename
    if path.is_file():
        df = pd.read_csv(path)
        print(f"{filename}: {df.shape}")


FINAL DELIVERABLE CHECK

PASS: RouteRanker_Module2_Feature_Engineering.ipynb
PASS: data/processed/route_features.csv
PASS: data/processed/route_profiles.csv
PASS: reports/module2_feature_dictionary.csv
PASS: reports/module2_data_quality_report.csv
PASS: reports/module2_final_validation_summary.csv
PASS: reports/module2_assumptions_and_limitations.md

DATASET SHAPES
data/processed/route_features.csv: (8322, 20)
data/processed/route_profiles.csv: (8322, 11)
